# SafeLens full feature tour with Qwen3-0.6B

This notebook is a detailed, end-to-end demonstration of the SafeLens Python library. It uses `Qwen/Qwen3-0.6B` as the target model for real model workflows, while also keeping lightweight dependency-free cells for APIs that do not require model weights.

The notebook covers:

- model adapter registry, static model inspection, and Qwen3 Dense validation;
- Qwen3 model loading, tokenization, generation, streaming generation, and TransformerLens-style wrapper helpers;
- hook points, temporary hooks, permanent hooks, caching hooks, `ActivationCache`, `HookedRoot`, and hook naming;
- mechanistic interpretability helpers: logits, losses, sampling, direct logit attribution, head scores, residual decomposition, logit lens, and SVD interpretation;
- TransformerLens/CircuitsVis-style visualizations for tokens, attention patterns, interactive attention browsers, activation patching grids and browsers, logit lens, neuron activations, next-token prediction browsers, token log-probs, top-k token/sample activations, and SafeLens reports;
- activation patching: SafeLens `PatchSpec`, generic patching, component patching, and every exported TransformerLens-style patch helper family;
- `FactoredMatrix`, `KeyValueCache`, tensor utilities, activation functions, device/tokenizer/HuggingFace utilities;
- pipeline configuration, method registry, dummy probe/monitor/attributor flow, reports, FlagSafe conversion, JSON schema, and CLI commands;
- an automated public API coverage audit for `SafeLens`, `SafeLens.core`, `SafeLens.utils`, and `SafeLens.viz`.

The real Qwen3 cells are guarded by `RUN_REAL_QWEN`. Set it to `True` after installing the model extras and ensuring the machine can download/load `Qwen/Qwen3-0.6B`.

## 0. Installation and execution mode

Run the bootstrap cell below before any `SafeLens` imports. It supports a cloned source checkout that has not been installed yet: it first adds the local `src/` directory to `sys.path`, then installs missing core dependencies or an editable package only when needed.

For the real Qwen3 workflow, set `SAFELENS_INSTALL_EXTRA = "models"` in the bootstrap cell and later set `RUN_REAL_QWEN = True`. The equivalent terminal command is:

```bash
python -m pip install -e ".[models]"
```

The base, dependency-light sections run with the core package dependencies (`pydantic`, `PyYAML`). The cells that need `torch`, `transformers`, `pandas`, or an actual model are guarded or marked explicitly.

In [1]:
# Bootstrap for a fresh machine or a notebook opened from the examples/ directory.
# Use comma-separated extras such as "models,viz" if you want native CircuitsVis too.
import importlib.util
import subprocess
import sys
from pathlib import Path

SAFELENS_INSTALL_EXTRA = "models"  # "" for core, "models", "viz", "models,viz", or "modelscope".
FORCE_EDITABLE_INSTALL = False


def _find_safelens_checkout(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "SafeLens").is_dir():
            return candidate
    return None


def _module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


PROJECT_ROOT = _find_safelens_checkout(Path.cwd().resolve())
if PROJECT_ROOT is not None:
    src_dir = PROJECT_ROOT / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

core_modules = ["pydantic", "yaml"]
extra_modules = {
    "models": ["torch", "transformers"],
    "modelscope": ["modelscope", "torch", "transformers"],
    "viz": ["circuitsvis"],
}
requested_extras = [
    extra.strip()
    for extra in SAFELENS_INSTALL_EXTRA.split(",")
    if extra.strip()
]
unsupported_extras = [extra for extra in requested_extras if extra not in extra_modules]
if unsupported_extras:
    raise ValueError(f"Unsupported SAFELENS_INSTALL_EXTRA values: {unsupported_extras!r}")

requested_extra_modules = sorted(
    {module for extra in requested_extras for module in extra_modules[extra]}
)
missing_modules = [
    name for name in ["SafeLens", *core_modules, *requested_extra_modules]
    if _module_missing(name)
]

if FORCE_EDITABLE_INSTALL or missing_modules:
    if PROJECT_ROOT is None:
        raise RuntimeError(
            "SafeLens is not importable and no local source checkout was found. "
            "Run this notebook from the SafeLens repository or install SafeLens manually first."
        )
    install_target = str(PROJECT_ROOT)
    if requested_extras:
        install_target = f"{install_target}[{','.join(requested_extras)}]"
    cmd = [sys.executable, "-m", "pip", "install", "-e", install_target, "--no-build-isolation"]
    print("Installing SafeLens because these modules are missing:", missing_modules)
    print(" ".join(cmd))
    subprocess.check_call(cmd)
    importlib.invalidate_caches()
else:
    print("SafeLens import path and requested dependencies are ready.")

print("SafeLens project root:", PROJECT_ROOT or "not found")
print("SafeLens install extra:", ",".join(requested_extras) or "core only")


SafeLens import path and requested dependencies are ready.
SafeLens project root: /workspace/SafeLens
SafeLens install extra: models


In [2]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
import contextlib
import io
import json
import math
import tempfile

import SafeLens
import SafeLens.core as sl_core
import SafeLens.utils as sl_utils
import SafeLens.viz as sl_viz

RUN_REAL_QWEN = True
MODEL_ID = "Qwen/Qwen3-0.6B"
CACHE_DIR = Path("../.cache/safelens/qwen3-0.6b-notebook").resolve()

print("SafeLens public symbols:", len(SafeLens.__all__))
print("SafeLens.core public symbols:", len(sl_core.__all__))
print("SafeLens.utils public symbols:", len(sl_utils.__all__))
print("SafeLens.viz public symbols:", len(sl_viz.__all__))
print("Real Qwen run enabled:", RUN_REAL_QWEN)

SafeLens public symbols: 209
SafeLens.core public symbols: 205
SafeLens.utils public symbols: 111
SafeLens.viz public symbols: 41
Real Qwen run enabled: True


In [3]:
from SafeLens import (
    ActivationCache,
    AttributionResult,
    FactoredMatrix,
    HookPoint,
    HookedRoot,
    KeyValueCache,
    KeyValueCacheEntry,
    MethodSpec,
    ModelLoadConfig,
    MonitoringSignal,
    OutputConfig,
    PatchSpec,
    PipelineConfig,
    PipelineSectionConfig,
    ProbeResult,
    RunReport,
    SVDInterpreter,
    SafetyReport,
    Slice,
    TokenAttribution,
    Visualization,
    colored_tokens,
    colored_tokens_multi,
    export_html,
    plot_activation_cache_browser,
    plot_activation_patching_browser,
    plot_activation_patching_grid,
    plot_attention_browser,
    plot_attention_heads,
    plot_attention_pattern,
    plot_attention_patterns,
    plot_bar,
    plot_cache_summary,
    plot_component_scores,
    plot_head_scores,
    plot_histogram,
    plot_line,
    plot_logit_lens,
    plot_model_performance,
    plot_next_token_browser,
    plot_neuron_activations,
    plot_scatter,
    plot_text_neuron_browser,
    plot_text_neuron_activations,
    plot_token_log_probs,
    plot_topk_samples,
    plot_topk_samples_browser,
    plot_topk_tokens,
    plot_topk_tokens_browser,
    render_run_report,
    render_safety_report,
    to_circuitsvis_attention_heads,
    to_circuitsvis_attention_pattern,
    to_circuitsvis_attention_patterns,
    to_circuitsvis_colored_tokens,
    to_circuitsvis_colored_tokens_multi,
    to_circuitsvis_model_performance,
    to_circuitsvis_text_neuron_activations,
    to_circuitsvis_token_log_probs,
    to_circuitsvis_topk_samples,
    to_circuitsvis_topk_tokens,
    attention_pattern_score,
    component_activation_patch,
    cross_entropy_loss,
    direct_logit_attribution,
    generic_activation_patch,
    get_act_patch_resid_pre,
    get_induction_head_detection_pattern,
    get_previous_token_head_detection_pattern,
    induction_attention_score,
    layer_pos_patch_setter,
    lm_accuracy,
    lm_cross_entropy_loss,
    logit_diff,
    logits_to_log_probs,
    make_patch_specs,
    patch_results_to_metric_grid,
    previous_token_attention_score,
    residual_stack_to_logits,
    run_activation_patch,
    softmax,
    test_prompt,
    topk_tokens,
)
from SafeLens.adapters.flagsafe_adapter import FlagSafeAdapter
from SafeLens.config import (
    ConfigValidationError,
    config_summary,
    format_pydantic_errors,
    iter_layer_refs,
    load_yaml_config,
    pipeline_config_json_schema,
    run_report_json_schema,
    validate_pipeline_config_file,
    validate_registered_methods,
    validate_static_hook_names,
    write_pipeline_config_json_schema,
)
from SafeLens.core import (
    cache_activations,
    get_act_name,
    make_cache_hook,
    matches_names_filter,
    safelens_act_name,
    temporary_hooks,
)
from SafeLens.core.registry import (
    RegistryError,
    create_attributor,
    create_monitor,
    create_probe,
    get_attributor,
    get_monitor,
    get_probe,
    list_attributors,
    list_monitors,
    list_probes,
    register_attributor,
    register_monitor,
    register_probe,
)
from SafeLens.probes.dummy import DummyProbe
from SafeLens.monitors.dummy import DummyMonitor
from SafeLens.attribution.dummy import DummyAttributor
from SafeLens.pipelines.runner import PipelineRunner, load_pipeline_config, run_from_config
from SafeLens.utils import (
    ArchitectureAdapter,
    ComponentHookSpec,
    ComponentRef,
    DummyModelWrapper,
    ModelAdapterCapabilities,
    ModelAdapterRegistry,
    ModelAdapterSpec,
    ModelDownloadPlan,
    Qwen3DenseModelWrapper,
    TransformerLensConfigView,
    TransformerLensCompatibleModelWrapper,
    architecture_adapter_for_model,
    architecture_adapter_for_name,
    build_model_wrapper,
    list_architecture_adapters,
    get_model_adapter_registry,
    is_supported_qwen3_dense_model_name,
    parse_qwen3_component_ref,
    qwen3_dense_size_billion,
    qwen3_hook_name_examples,
    qwen3_supported_hook_components,
    resolve_model_download_plan,
    resolve_transformer_lens_compatible_model_name,
    supported_transformer_component_names,
    transformer_lens_model_kind,
    transformer_lens_official_model_names,
    validate_qwen3_dense_model_name,
    validate_qwen3_hook_ref,
)

print("Imports loaded")
import SafeLens.utils.model_bridge as model_bridge
from SafeLens.cli import build_parser, main as safelens_cli_main


Imports loaded


## 1. Model registry, Qwen3-0.6B support, and static inspection

The model registry lets a workflow inspect support and download plans without loading model weights. This is useful for notebooks, CI, and dry-run configuration validation.

In [4]:
registry = get_model_adapter_registry()
adapter_rows = registry.list_supported()
print("Registered adapter sources:", registry.source_names())
pprint(adapter_rows)

inspection = registry.inspect_model(MODEL_ID)
print("\nStatic inspection for", MODEL_ID)
pprint(inspection)

qwen_config = ModelLoadConfig(
    source="qwen3_dense",
    name=MODEL_ID,
    dtype="bfloat16",
    device="cuda",
    cache_dir=str(CACHE_DIR),
    trust_remote_code=True,
)
download_plan = resolve_model_download_plan(qwen_config)
print("\nDownload plan:")
pprint(download_plan.to_dict())

Registered adapter sources: ['dummy', 'hf', 'hooked_transformer', 'huggingface', 'local', 'mock', 'modelscope', 'ms', 'none', 'qwen3', 'qwen3_dense', 'tl', 'transformer_lens', 'transformerlens']
[{'aliases': ['mock', 'none'],
  'capabilities': {'cache_policy': 'no external cache',
                   'notes': ['Does not download or execute model code.'],
                   'supported_hooks': ['integer layer refs',
                                       'string layer refs'],
                   'supported_patches': ['replace', 'add'],
                   'supports_attention_pattern': False,
                   'supports_attention_scores': False,
                   'supports_local_path': False,
                   'supports_remote_download': False},
  'dependencies': [],
  'description': 'In-memory adapter for tests, CI, and architecture demos.',
  'display_name': 'Dummy',
  'model_name_patterns': ['dummy', 'mock', 'none'],
  'name': 'dummy'},
 {'aliases': ['hf'],
  'capabilities': {'cache_po

In [5]:
print("Qwen3 dense size:", qwen3_dense_size_billion(MODEL_ID))
print("Supported dense Qwen3:", is_supported_qwen3_dense_model_name(MODEL_ID))
validate_qwen3_dense_model_name(MODEL_ID)

print("Supported Qwen3 components:")
print(qwen3_supported_hook_components(include_attention=True))

print("Hook name examples:")
for name in qwen3_hook_name_examples():
    print("-", name, "=>", parse_qwen3_component_ref(name))
    validate_qwen3_hook_ref(name)

adapter = architecture_adapter_for_name(model_name=MODEL_ID)
print("\nArchitecture adapter:", adapter.name)
pprint(adapter.inspect())
print("All supported transformer component names:")
print(supported_transformer_component_names(include_attention=True))


Qwen3 dense size: 0.6
Supported dense Qwen3: True
Supported Qwen3 components:
['attn_out', 'k', 'mlp_out', 'post', 'pre', 'pre_linear', 'q', 'resid_mid', 'resid_post', 'resid_pre', 'result', 'v', 'z', 'attn_scores', 'pattern']
Hook name examples:
- layer_0.resid_pre => (0, 'resid_pre')
- layer_0.resid_mid => (0, 'resid_mid')
- layer_0.resid_post => (0, 'resid_post')
- layer_0.attn_out => (0, 'attn_out')
- layer_0.mlp_out => (0, 'mlp_out')
- layer_0.pre => (0, 'pre')
- layer_0.pre_linear => (0, 'pre_linear')
- layer_0.post => (0, 'post')
- layer_0.q => (0, 'q')
- layer_0.k => (0, 'k')
- layer_0.v => (0, 'v')
- layer_0.z => (0, 'z')
- layer_0.result => (0, 'result')
- layer_0.pattern => (0, 'pattern')
- layer_0.attn_scores => (0, 'attn_scores')
- blocks.0.hook_resid_pre => (0, 'resid_pre')
- blocks.0.attn.hook_q => (0, 'q')
- blocks.0.mlp.hook_pre => (0, 'pre')
- blocks.0.mlp.hook_pre_linear => (0, 'pre_linear')
- blocks.0.mlp.hook_post => (0, 'post')
- blocks.0.attn.hook_result => (0, '

## 1b. Architecture bridge and component-level helper surface

SafeLens model wrappers use an architecture bridge to translate model-specific module layouts into canonical components such as `resid_pre`, `q`, `k`, `v`, `pattern`, and `mlp_out`. The first cell below shows the public bridge dataclasses and adapter metadata for `Qwen/Qwen3-0.6B`; the second cell exercises the dependency-free tensor-shape helpers that the bridge uses for QKV splitting/merging and attention result transforms.


In [6]:
print("Bridge dataclasses:")
component_ref = ComponentRef(layer=0, component="q", original="blocks.0.attn.hook_q")
component_spec = ComponentHookSpec(
    component="demo",
    mode="forward_output",
    module_paths=("model.layers.{layer}.demo",),
    aliases=("demo_alias",),
)
print(component_ref.safelens_name, component_ref.transformer_lens_name)
print(component_spec.all_names())

print("Registered architecture adapters:")
for row in list_architecture_adapters():
    print(row["name"], "components=", len(row["target_components"]), "notes=", row["notes"][:1])

adapter_by_name = architecture_adapter_for_name(model_name=MODEL_ID)
print("Selected adapter for Qwen3-0.6B:", adapter_by_name.name)
print("parse refs:")
for ref in [0, "layer_0.q", "blocks.0.attn.hook_q", ("q", 0, "attn")]:
    parsed = adapter_by_name.parse_component_ref(ref)
    print(ref, "=>", None if parsed is None else (parsed.safelens_name, parsed.transformer_lens_name))

class TinyConfig:
    model_type = "qwen2"

class TinyModelForAdapter:
    config = TinyConfig()

adapter_by_model = architecture_adapter_for_model(TinyModelForAdapter(), model_name=MODEL_ID)
print("Selected adapter from loaded-model config:", adapter_by_model.name)
print("Qwen routed MoE check:", model_bridge.is_qwen_routed_moe_model_name("Qwen/Qwen3-30B-A3B"))
print("TransformerLens names:", [model_bridge.transformer_lens_component_name(name, 0) for name in ["resid_pre", "q", "pre", "ln1_scale"]])


Bridge dataclasses:
layer_0.q blocks.0.attn.hook_q
('demo', 'demo_alias')
Registered architecture adapters:
mamba2_ssm components= 9 notes= ['State-space adapter for HuggingFace Mamba2 models. It exposes residual, normalization, and mixer projection hooks; attention hooks are unsupported.']
mamba_ssm components= 10 notes= ['State-space adapter for HuggingFace Mamba models. It exposes residual, normalization, and mixer projection hooks; attention hooks are unsupported.']
routed_moe_decoder components= 19 notes= ['Covers routed MoE decoder families with standard q/k/v/o attention projections; dense neuron-level MLP matrices are intentionally not exposed.']
llama_like_decoder components= 19 notes= ['Covers RoPE decoder families with model.layers or model.language_model.layers and q/k/v/o projections.']
apertus_decoder components= 19 notes= ['Apertus follows LLaMA-style q/k/v/o projections but uses attention_layernorm/feedforward_layernorm names and an ungated MLP.']
gpt_oss_decoder compon

In [7]:
print("Component vocabulary samples:")
print("patch/cache components:", supported_transformer_component_names()[:12])
print("with attention:", supported_transformer_component_names(include_attention=True)[-12:])

class TinyAttentionConfig:
    model_type = "qwen2"
    num_attention_heads = 2
    num_key_value_heads = 1
    hidden_size = 4
    head_dim = 2

class TinyAttentionModel:
    config = TinyAttentionConfig()

qkv_activation = [[[10, 11, 12, 13, 20, 21, 30, 31]]]
q_spec = ComponentHookSpec(
    component="q",
    mode="forward_output",
    module_paths=("unused",),
    activation="split_qkv_heads",
    qkv_layout="split",
)
print("split_qkv_slice_bounds:", model_bridge.split_qkv_slice_bounds(qkv_activation, q_heads=2, kv_heads=1))
print("split_qkv_slices:", model_bridge.split_qkv_slices(qkv_activation, q_heads=2, kv_heads=1))
print("q bounds:", model_bridge.qkv_weight_bounds(2, q_heads=2, kv_heads=1))
print("split heads:", model_bridge.split_heads([[[1, 2, 3, 4]]], n_heads=2))
print("merge heads:", model_bridge.merge_heads([[[[1, 2], [3, 4]]]], [[[0, 0, 0, 0]]]))
print("split qkv heads:", model_bridge.split_qkv_heads(qkv_activation, TinyAttentionModel(), q_spec))
print("merge qkv heads:", model_bridge.merge_qkv_heads([[[[99, 98], [97, 96]]]], qkv_activation, TinyAttentionModel(), q_spec))
print("split interleaved qkv heads:", model_bridge.split_interleaved_qkv_heads([[[1, 2, 5, 6, 7, 8, 3, 4]]], "k", q_heads=2, kv_heads=1))
print("merge interleaved qkv heads:", model_bridge.merge_interleaved_qkv_heads([[[[55, 56]]]], [[[1, 2, 5, 6, 7, 8, 3, 4]]], "k", q_heads=2, kv_heads=1))

print("attention helpers:")
print("q heads:", model_bridge.attention_head_count(TinyAttentionModel()))
print("qkv group size:", model_bridge.qkv_group_size(q_heads=8, kv_heads=2))
print("key-value heads fallback:", model_bridge.key_value_head_count(TinyAttentionModel()))
print("head count for component:", model_bridge.head_count_for_component(TinyAttentionModel(), "result"))
print("attention head dim:", model_bridge.attention_head_dim(TinyAttentionModel()))
print("first attention head:", model_bridge.first_attention_head([[[[1, 2], [3, 4]]]]))
print("sum heads:", model_bridge.sum_attention_heads([[[[1, 2], [3, 4]]]]))
print("zeros for bias:", model_bridge.zeros_for_attention_bias(TinyAttentionModel(), "q"))
print("zeros last dim:", model_bridge.zeros_like_last_dim([[1, 2, 3], [4, 5, 6]]))
print("transpose weight:", model_bridge.transpose_2d_weight([[1, 2], [3, 4]]))
print("clone/add/subtract:", model_bridge.clone_tensor_like([1, 2]), model_bridge.add_values([1, 2], [3, 4]), model_bridge.subtract_values([3, 4], [1, 1]))
class TinyNorm:
    weight = [2.0, 3.0]
    bias = [0.5, -0.5]
print("norm helpers:", model_bridge.apply_norm_affine(TinyNorm(), [[1.0, 2.0]]))


Component vocabulary samples:
patch/cache components: ('resid_pre', 'resid_mid', 'resid_post', 'attn_in', 'attn_out', 'mlp_in', 'mlp_out', 'q_input', 'k_input', 'v_input', 'pre', 'pre_linear')
with attention: ('decoder_ln1_scale', 'decoder_ln2_scale', 'decoder_ln3_scale', 'decoder_ln1_normalized', 'decoder_ln2_normalized', 'decoder_ln3_normalized', 'ssm_in', 'ssm_conv', 'ssm_x', 'ssm_dt', 'ssm_out', 'ssm_inner_norm')
split_qkv_slice_bounds: ((0, 4), (4, 6), (6, 8))
split_qkv_slices: ([[[10, 11, 12, 13]]], [[[20, 21]]], [[[30, 31]]])
q bounds: {'q': (0, 4), 'k': (4, 6), 'v': (6, 8)}
split heads: [[[[1, 2], [3, 4]]]]
merge heads: [[[1, 2, 3, 4]]]
split qkv heads: [[[[10, 11], [12, 13]]]]
merge qkv heads: [[[99, 98, 97, 96, 20, 21, 30, 31]]]
split interleaved qkv heads: [[[[7, 8]]]]
merge interleaved qkv heads: [[[1, 2, 5, 6, 55, 56, 3, 4]]]
attention helpers:
q heads: 2
qkv group size: 4
key-value heads fallback: 1
head count for component: 2
attention head dim: 2
first attention head: [

## 2. Build or load the Qwen3-0.6B wrapper

The code below constructs the wrapper in all modes. Real weight loading happens only when `RUN_REAL_QWEN=True`.

In [8]:
qwen_wrapper = build_model_wrapper(qwen_config)
print(type(qwen_wrapper).__name__, qwen_wrapper.name)

if RUN_REAL_QWEN:
    model = qwen_wrapper.load_model()
    print("Loaded model type:", type(model).__name__)
    print("cfg:", qwen_wrapper.cfg.to_dict())
else:
    print("Skipped real model load. Set RUN_REAL_QWEN=True to download/load Qwen3-0.6B.")

dummy_wrapper = build_model_wrapper(ModelLoadConfig(source="dummy", name="dummy"))
dummy_wrapper.load_model()
print("Dummy wrapper is available for lightweight cells:", type(dummy_wrapper).__name__)

Qwen3DenseModelWrapper Qwen/Qwen3-0.6B


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 17683.33it/s]

Loaded model type: Qwen3ForCausalLM
cfg: {'model_name': 'Qwen/Qwen3-0.6B', 'model_type': 'qwen3', 'n_layers': 28, 'n_heads': 16, 'n_key_value_heads': 8, 'd_model': 1024, 'd_head': 128, 'd_vocab': 151936, 'n_ctx': 40960, 'd_mlp': 3072, 'act_fn': 'silu', 'normalization_type': 'RMS', 'positional_embedding_type': 'rotary', 'device': 'cuda', 'dtype': 'bfloat16', 'original_architecture': 'Qwen3ForCausalLM', 'use_attn_result': False, 'use_split_qkv_input': False, 'use_hook_mlp_in': False, 'use_attn_in': False, 'ungroup_grouped_query_attention': False, 'attn_only': False, 'parallel_attn_mlp': False, 'rmsnorm_uses_offset': False}
Dummy wrapper is available for lightweight cells: DummyModelWrapper


In [9]:
if RUN_REAL_QWEN:
    prompt = "Explain activation patching in one sentence:"
    tokens = qwen_wrapper.to_tokens(prompt, prepend_bos=True)
    print("tokens shape:", getattr(tokens, "shape", None))
    print("str tokens:", qwen_wrapper.to_str_tokens(prompt)[:12])
    print("round trip:", qwen_wrapper.to_string(tokens[0]))
    print("single token id:", qwen_wrapper.to_single_token(" safety"))
    print("single token text:", qwen_wrapper.to_single_str_token(qwen_wrapper.to_single_token(" safety")))
    print("position of token:", qwen_wrapper.get_token_position(" activation", prompt))
    generated = qwen_wrapper.generate(prompt, max_new_tokens=24, do_sample=False)
    print("generated:", generated)
    print("stream chunks:")
    for chunk in qwen_wrapper.generate_stream(prompt, max_new_tokens=24, do_sample=False, max_tokens_per_yield=8):
        print(repr(chunk))
else:
    print(dummy_wrapper.generate("SafeLens"))
    print(list(dummy_wrapper.generate_stream("SafeLens", max_tokens_per_yield=8)))

tokens shape: torch.Size([1, 9])
str tokens: ['Ex', 'plain', ' activation', ' patch', 'ing', ' in', ' one', ' sentence', ':']
round trip: Explain activation patching in one sentence:
single token id: 7149
single token text:  safety
position of token: 2


generated: Explain activation patching in one sentence: "Activation patching is a method of modifying the software to allow for the use of a new feature or function that is
stream chunks:
' "Activation patching is a method of'


' modifying the software to allow for the use'
' of a new feature or function that is'


## 3. Hook points, HookedRoot, and ActivationCache

SafeLens mirrors the most useful TransformerLens hook mechanics without depending on TransformerLens: named `HookPoint`s, temporary and permanent hooks, cache hooks, name filters, alias resolution, and an `ActivationCache` with logit-lens style helpers.

In [10]:
hp = HookPoint("blocks.0.hook_resid_pre")

def add_one(activation, hook):
    hook.ctx["seen"] = hook.ctx.get("seen", 0) + 1
    return [value + 1 for value in activation]

handle = hp.add_hook(add_one)
print("hook output:", hp([1, 2, 3]))
print("hook context:", hp.ctx)
print("has hooks:", hp.has_hooks())
handle.remove()
print("after remove:", hp([1, 2, 3]), hp.has_hooks())

print("TL name q2:", get_act_name("q", 2))
print("TL compact q2:", get_act_name("q2"))
print("SafeLens name:", safelens_act_name("resid_post", 3))
print("filter exact/alias:", matches_names_filter("blocks.0.attn.hook_q", "layer_0.q"))

hook output: [2, 3, 4]
hook context: {'seen': 1}
has hooks: True
after remove: [1, 2, 3] False
TL name q2: blocks.2.attn.hook_q
TL compact q2: blocks.2.attn.hook_q
SafeLens name: layer_3.resid_post
filter exact/alias: True


In [11]:
root = HookedRoot("toy-root")
resid = root.add_hook_point("blocks.0.hook_resid_pre")
mlp = root.add_hook_point("blocks.0.hook_mlp_out")

cache, fwd_hooks, bwd_hooks = root.get_caching_hooks(
    lambda name: name.endswith("resid_pre") or name.endswith("mlp_out"),
    remove_batch_dim=False,
)
with root.hooks(fwd_hooks=fwd_hooks, bwd_hooks=bwd_hooks):
    resid([[1.0, 2.0]])
    mlp([[3.0, 4.0]])

print(cache)
print("cache keys:", list(cache.keys()))
print("tuple alias access:", cache["resid_pre", 0])
print("selected keys:", list(cache.select("blocks.0.hook_mlp_out").keys()))
print("cache as dict:", cache.to_dict())

manual_cache = ActivationCache({"blocks.0.hook_resid_pre": [[1, 2], [3, 4]]})
manual_cache.store("blocks.1.hook_resid_post", [[5, 6]], detach=False)
print("manual cache keys:", list(manual_cache.keys()))
print("keys matching resid:", manual_cache.keys_matching(lambda name: "resid" in name))
print("clone dict:", manual_cache.clone().to_dict())
print("batch slice:", manual_cache.apply_slice_to_batch_dim(0).to_dict())

ActivationCache(2 activations, with batch dim)
cache keys: ['blocks.0.hook_resid_pre', 'blocks.0.hook_mlp_out']
tuple alias access: [[1.0, 2.0]]
selected keys: ['blocks.0.hook_mlp_out']
cache as dict: {'blocks.0.hook_resid_pre': [[1.0, 2.0]], 'blocks.0.hook_mlp_out': [[3.0, 4.0]]}
manual cache keys: ['blocks.0.hook_resid_pre', 'blocks.1.hook_resid_post']
keys matching resid: ['blocks.0.hook_resid_pre', 'blocks.1.hook_resid_post']
clone dict: {'blocks.0.hook_resid_pre': [[1, 2], [3, 4]], 'blocks.1.hook_resid_post': [[5, 6]]}
batch slice: {'blocks.0.hook_resid_pre': [1, 2], 'blocks.1.hook_resid_post': [5, 6]}


In [12]:
if RUN_REAL_QWEN:
    prompt = "The capital of France is"
    tokens = qwen_wrapper.to_tokens(prompt)
    logits, cache = qwen_wrapper.run_with_cache(
        tokens,
        names_filter=lambda name: name.endswith("hook_resid_post") or name.endswith("hook_mlp_out"),
        return_cache_object=True,
        remove_batch_dim=False,
    )
    print("logits shape:", logits.shape)
    print("cached keys sample:", list(cache.keys())[:10])
    print("resid post layer 0 shape:", cache["resid_post", 0].shape)

    def zero_last_pos(activation, hook):
        patched = activation.clone()
        patched[:, -1, :] = 0
        return patched

    patched_logits = qwen_wrapper.run_with_hooks(
        tokens,
        fwd_hooks=[("blocks.0.hook_resid_post", zero_last_pos)],
        return_type="logits",
    )
    print("patched logits shape:", patched_logits.shape)
else:
    print("Qwen cache/hook cell skipped. Enable RUN_REAL_QWEN for real activations.")

logits shape: torch.Size([1, 5, 151936])
cached keys sample: ['blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_mlp_out', 'blocks.1.hook_resid_post', 'blocks.2.hook_mlp_out', 'blocks.2.hook_resid_post', 'blocks.3.hook_mlp_out', 'blocks.3.hook_resid_post', 'blocks.4.hook_mlp_out', 'blocks.4.hook_resid_post']
resid post layer 0 shape: torch.Size([1, 5, 1024])
patched logits shape: torch.Size([1, 5, 151936])


## 4. Tensor helpers, activation functions, losses, and sampling

These helpers are deliberately backend-flexible. They work with Python nested lists, NumPy arrays, or torch tensors when those libraries are installed.

In [13]:
values = [[1.0, 2.0, 3.0], [2.0, 0.0, -1.0]]
print("softmax:", softmax(values))
print("log probs:", logits_to_log_probs(values))
print("topk:", topk_tokens(values[0], k=2))
print("sample deterministic temperature=0:", sl_core.sample_logits(values[0], temperature=0.0))

logits = [[[0.0, 2.0, 1.0], [1.0, 0.0, 3.0], [3.0, 1.0, 0.0]]]
tokens = [[1, 2, 0]]
print("cross entropy:", cross_entropy_loss(logits[0], tokens[0]))
print("LM per-token CE:", lm_cross_entropy_loss(logits, tokens, per_token=True))
print("LM CE:", lm_cross_entropy_loss(logits, tokens))
print("LM accuracy:", lm_accuracy(logits, tokens))
print("logit diff:", logit_diff(logits, correct_token=0, incorrect_token=2, pos=-1))

print("Slice int:", Slice(1).apply([[1, 2, 3], [4, 5, 6]], dim=1))
print("Slice unwrap:", Slice.unwrap(1).apply([[1, 2, 3], [4, 5, 6]], dim=1))
print("Slice bool:", Slice([True, False, True]).apply([[1, 2, 3], [4, 5, 6]], dim=1))
print("indices:", Slice((0, 3, 2)).indices(5).tolist())
print("to_numpy facade:", sl_core.to_numpy([1, 2, 3]).tolist())
print("transpose:", sl_core.transpose([[1, 2], [3, 4]]))
print("triangular/square:", sl_core.is_lower_triangular([[1, 0], [2, 3]]), sl_core.is_square([[1, 2], [3, 4]]))
print("cumsum:", sl_core.get_cumsum_along_dim([[1, 2, 3]], dim=1))
print("offset position ids:", sl_core.get_offset_position_ids(1, [[0, 1, 1, 1]]))
print("repeat heads:", sl_core.repeat_along_head_dimension([[[1, 2]]], 2))
print("filter prefix:", sl_core.filter_dict_by_prefix({"a.x": 1, "b.y": 2}, "a"))

softmax: [[0.09003057317038046, 0.24472847105479764, 0.6652409557748218], [0.8437947344813395, 0.11419519938459449, 0.04201006613406605]]
log probs: [[-2.40760596444438, -1.4076059644443804, -0.40760596444438046], [-0.16984601955628567, -2.1698460195562856, -3.1698460195562856]]
topk: ([2, 1], [3.0, 2.0])
sample deterministic temperature=0: [2]
cross entropy: 0.24909933451898394
LM per-token CE: [[1.4076059644443804, 2.1698460195562856]]
LM CE: 1.7887259920003329
LM accuracy: 0.0
logit diff: 3.0
Slice int: [2, 5]
Slice unwrap: [[2], [5]]
Slice bool: [[1, 3], [4, 6]]
indices: [0, 2]
to_numpy facade: [1, 2, 3]
transpose: [[1, 3], [2, 4]]
triangular/square: True True
cumsum: [[1, 3, 6]]
offset position ids: [[0, 1, 2]]
repeat heads: [[[[1, 2], [1, 2]]]]
filter prefix: {'x': 1}


In [14]:
activation_inputs = [-2.0, -0.5, 0.0, 0.5, 2.0]
for name, fn in sl_core.SUPPORTED_ACTIVATIONS.items():
    with contextlib.suppress(Exception):
        print(name, [round(float(x), 6) for x in fn(activation_inputs)])

print("Named activation helpers:")
for fn in [sl_core.relu, sl_core.gelu, sl_core.gelu_new, sl_core.gelu_fast, sl_core.gelu_pytorch_tanh, sl_core.silu, sl_core.solu, sl_core.xielu]:
    with contextlib.suppress(Exception):
        print(fn.__name__, fn(activation_inputs))

solu [-0.025109, -0.028133, 0.0, 0.076474, 1.370928]
solu_ln [-0.025109, -0.028133, 0.0, 0.076474, 1.370928]
gelu_new [-0.045402, -0.154286, 0.0, 0.345714, 1.954598]
gelu_fast [-0.045402, -0.154286, 0.0, 0.345714, 1.954598]
silu [-0.238406, -0.18877, 0.0, 0.31123, 1.761594]
relu [0.0, 0.0, 0.0, 0.5, 2.0]
gelu [-0.0455, -0.154269, 0.0, 0.345731, 1.9545]
gelu_pytorch_tanh [-0.045402, -0.154286, 0.0, 0.345714, 1.954598]
xielu [-0.091732, -0.164775, -1e-06, 0.45, 4.2]
Named activation helpers:
relu [0.0, 0.0, 0.0, 0.5, 2.0]
gelu [-0.04550026389635842, -0.15426876936299344, 0.0, 0.34573123063700656, 1.9544997361036416]
gelu_new [-0.04540230591222494, -0.15428599017485606, 0.0, 0.34571400982514394, 1.954597694087775]
gelu_fast [-0.04540230591282446, -0.15428599017516514, 0.0, 0.34571400982483486, 1.9545976940871754]
gelu_pytorch_tanh [-0.04540230591222494, -0.15428599017485606, 0.0, 0.34571400982514394, 1.954597694087775]
silu [-0.2384058440442351, -0.1887703343990727, 0.0, 0.311229665600927

## 5. Mechanistic interpretability helpers

This section demonstrates direct logit attribution, residual-to-logit projection, attention-pattern scores, head-detection patterns, ablation hooks, and prompt testing.

In [15]:
residual_stack = [
    [1.0, 0.0],
    [0.5, 0.5],
    [0.0, 1.0],
]
unembed = [[2.0, -1.0, 0.0], [0.0, 1.0, 2.0]]
token_direction = [1.0, -1.0]
print("residual stack -> logits:", residual_stack_to_logits(residual_stack, unembed))
print("direct logit attribution:", direct_logit_attribution(residual_stack, token_direction))

z = [[[[1.0, 2.0], [3.0, 4.0]]]]
W_O = [[[1.0, 0.0, 0.5], [0.0, 1.0, 0.5]], [[1.0, 1.0, 0.0], [0.5, 0.0, 1.0]]]
print("head results from z:", sl_core.compute_head_results_from_z(z, W_O))

pattern = [[[[1.0, 0.0, 0.0], [0.8, 0.2, 0.0], [0.1, 0.7, 0.2]]]]
print("previous-token score:", previous_token_attention_score(pattern))
print("induction score:", induction_attention_score(pattern))
print("generic pattern score offset=-1:", attention_pattern_score(pattern, offset=-1))
tokens_for_heads = [1, 2, 1, 2, 3]
previous_pattern = get_previous_token_head_detection_pattern(tokens_for_heads)
duplicate_pattern = sl_core.get_duplicate_token_head_detection_pattern(tokens_for_heads)
induction_pattern = get_induction_head_detection_pattern(tokens_for_heads)
print("previous-token detection pattern:", previous_pattern)
print("induction detection pattern:", induction_pattern)
print("duplicate-token detection pattern:", duplicate_pattern)
print("supported head detectors:", sl_core.get_supported_heads())
print("head similarity:", sl_core.compute_head_attention_similarity_score(previous_pattern, previous_pattern, exclude_bos=False, exclude_current_token=False, error_measure="mul"))

residual stack -> logits: [[2.0, -1.0, 0.0], [1.0, 0.0, 1.0], [0.0, 1.0, 2.0]]
direct logit attribution: [1.0, 0.0, -1.0]
head results from z: [[[[1.0, 2.0, 1.5], [5.0, 3.0, 4.0]]]]
previous-token score: [[0.75]]
induction score: [[0.75]]
generic pattern score offset=-1: [[0.75]]
previous-token detection pattern: [[0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0]]
induction detection pattern: [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]]
duplicate-token detection pattern: [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0]]
Supported heads: ['previous_token_head', 'duplicate_token_head', 'induction_head']
supported head detectors: ['previous_token_head', 'duplicate_token_head', 'induction_head']
head similarity: 1.0


In [16]:
activation = [[1.0, 2.0], [3.0, 4.0]]
print("zero ablation:", sl_core.zero_ablation_hook(activation))
print("mean ablation:", sl_core.mean_ablation_hook(activation))
replace_hook = sl_core.replace_activation_hook([[9.0, 9.0], [9.0, 9.0]])
print("replace activation:", replace_hook(activation))

if RUN_REAL_QWEN:
    prompt_result = test_prompt(
        qwen_wrapper,
        "The capital of France is",
        " Paris",
        incorrect_token=" London",
        print_details=True,
    )
    pprint({k: v for k, v in prompt_result.items() if k != "logits"})
else:
    print("test_prompt against Qwen skipped until RUN_REAL_QWEN=True")

zero ablation: [[0, 0], [0, 0]]
mean ablation: [[2.5, 2.5], [2.5, 2.5]]
replace activation: [[9.0, 9.0], [9.0, 9.0]]
Prompt: 'The capital of France is'
Predicted token: ' Paris' (id 12095)
Correct token: ' Paris' (id 12095)
Logit diff: 6.0000
Top tokens:
  ' Paris' (12095): 17.5000
  ' located' (7407): 14.3750
  ' the' (279): 14.1250
  '...' (1112): 13.7500
  ' ____' (30743): 13.6875
  ' ______' (32671): 13.6250
  ' ' (220): 13.5625
  ' in' (304): 13.3750
  '____' (2130): 13.2500
  ' a' (264): 13.1875
{'correct_logit': 17.5,
 'correct_token': ' Paris',
 'correct_token_id': 12095,
 'incorrect_logit': 11.5,
 'incorrect_token': ' London',
 'incorrect_token_id': 7148,
 'is_correct': True,
 'logit_diff': 6.0,
 'predicted_token': ' Paris',
 'predicted_token_id': 12095,
 'prompt': 'The capital of France is',
 'top_tokens': [{'logit': 17.5, 'token': ' Paris', 'token_id': 12095},
                {'logit': 14.375, 'token': ' located', 'token_id': 7407},
                {'logit': 14.125, 'token':

## 6. FactoredMatrix and SVDInterpreter

`FactoredMatrix` is used for QK/OV circuits and composition scores without eagerly materializing every dense product. `SVDInterpreter` projects singular directions through the unembedding, matching TransformerLens-style circuit interpretation.

In [17]:
fm = FactoredMatrix([[1.0, 2.0], [0.0, 1.0]], [[2.0, 0.0], [1.0, 3.0]])
print(fm)
print("pair:", fm.pair)
print("shape/ndim/ldim/rdim/mdim:", fm.shape, fm.ndim, fm.ldim, fm.rdim, fm.mdim)
print("AB:", fm.AB)
print("BA:", fm.BA)
print("T.AB:", fm.T.AB)
print("matmul vector:", fm @ [1.0, 1.0])
print("left matmul vector:", [1.0, 1.0] @ fm)
print("scaled AB:", (0.5 * fm).AB)
print("corner:", fm.get_corner(1))
print("norm:", fm.norm())
u, s, v = fm.svd()
print("svd shapes:", sl_core.get_corner(u, 2), s, sl_core.get_corner(v, 2))
print("collapse_l:", fm.collapse_l())
print("collapse_r:", fm.collapse_r())
print("make_even AB:", fm.make_even().AB)
identity_factor = FactoredMatrix([[[1, 0], [0, 1]]], [[[1, 0], [0, 1]]])
print("composition scores:", sl_core.composition_scores(identity_factor, identity_factor))

FactoredMatrix: Shape((2, 2)), Hidden Dim(2)
pair: ([[1.0, 2.0], [0.0, 1.0]], [[2.0, 0.0], [1.0, 3.0]])
shape/ndim/ldim/rdim/mdim: (2, 2) 2 2 2 2
AB: [[4.0, 6.0], [1.0, 3.0]]
BA: [[2.0, 4.0], [1.0, 5.0]]
T.AB: [[4.0, 1.0], [6.0, 3.0]]
matmul vector: [10.0, 4.0]
left matmul vector: [5.0, 9.0]
scaled AB: [[2.0, 3.0], [0.5, 1.5]]
corner: [[4.0]]
norm: 7.874007874011811
svd shapes: [[-0.9193681924785362, -0.3933981782605884], [-0.3933981782605885, 0.919368192478536]] [7.836696539454048, 0.7656287275885756] [[-0.5194626240380532, -0.8544931727214091], [-0.8544931727214091, 0.5194626240380532]]
collapse_l: [[-4.070870948174731, -6.6964036896529775], [-0.6542245205638174, 0.3977155078720773]]
collapse_r: [[-7.2048095324806685, -0.30119694665731794], [-3.0829421422022807, 0.7038946993927502]]
make_even AB: [[4.0, 5.999999999999998], [1.0000000000000004, 2.999999999999999]]
composition scores: [[0.7071067811865475]]


In [18]:
class TinySVDModel:
    class cfg:
        n_layers = 1
        n_heads = 1
        d_vocab = 2

    def tl_parameters(self):
        return {
            "blocks.0.attn.W_V": [[[2.0, 0.0], [0.0, 1.0]]],
            "blocks.0.attn.W_O": [[[1.0, 0.0], [0.0, 1.0]]],
            "blocks.0.mlp.W_in": [[2.0, 1.0], [1.0, 2.0]],
            "blocks.0.mlp.W_out": [[2.0, 1.0], [1.0, 2.0]],
            "unembed.W_U": [[1.0, 0.0], [0.0, 1.0]],
        }

svd_interpreter = SVDInterpreter(TinySVDModel())
print("OV singular vectors:", svd_interpreter.get_singular_vectors("OV", 0, head_index=0, num_vectors=2))
print("w_in singular vectors:", svd_interpreter.get_singular_vectors("w_in", 0, num_vectors=2))
print("w_out singular vectors:", svd_interpreter.get_singular_vectors("w_out", 0, num_vectors=2))

if RUN_REAL_QWEN:
    real_svd = SVDInterpreter(qwen_wrapper)
    print(real_svd.get_singular_vectors("OV", layer_index=0, head_index=0, num_vectors=3).shape)
else:
    print("Real Qwen SVDInterpreter skipped until RUN_REAL_QWEN=True")

OV singular vectors: [[[1.0, 0.0]], [[0.0, 1.0]]]
w_in singular vectors: [[[-0.7071067811865476, -0.7071067811865475]], [[-0.7071067811865475, 0.7071067811865476]]]
w_out singular vectors: [[[-0.7071067811865476, -0.7071067811865475]], [[-0.7071067811865475, 0.7071067811865476]]]


torch.Size([151936, 1, 3])


## 7. Activation patching, including the full exported helper family

SafeLens exposes low-level patch specs, generic patch runners, component-level patch grids, TransformerLens-style patch setters, and a full family of `get_act_patch_*` helpers. The first cell executes with the dependency-free dummy wrapper; the Qwen3 cell is guarded by `RUN_REAL_QWEN`.

In [19]:
clean_cache = ActivationCache({"layer_0": {"value": 10.0, "batch": {"text": "clean"}}})

def patch_metric(output):
    return float(output.get("risk_score", 0.0))

def dict_replace_setter(corrupted_activation, spec, clean_cache):
    return dict(clean_cache[spec.clean_name])

spec = PatchSpec(layer=0, activation_name="layer_0", setter=dict_replace_setter)
patch_result = run_activation_patch(
    dummy_wrapper,
    {"text": "corrupted", "risk_score": 0.25},
    clean_cache,
    spec,
    patch_metric,
)
print("single patch result:", patch_result.metric, patch_result.output, patch_result.spec)

generic_results = generic_activation_patch(
    dummy_wrapper,
    {"text": "corrupted", "risk_score": 0.8},
    clean_cache,
    [spec],
    patch_metric,
)
print("generic detailed result count:", len(generic_results))
print("metric grid:", patch_results_to_metric_grid(generic_results))
print("index table:", sl_core.patch_results_to_index_table(generic_results))

activation = [[[1, 2], [3, 4]]]
activation_cache = ActivationCache({"layer_0.resid_pre": [[[9, 9], [8, 8]]]})
pos_spec = PatchSpec(layer="layer_0.resid_pre", activation_name="layer_0.resid_pre", target_index=(0, 1))
print("apply_patch replace:", sl_core.apply_patch(activation, pos_spec, activation_cache))
print("make_patch_specs:", make_patch_specs([0, 1], activation_name="layer_0"))
print("make_patch_hook callable:", callable(sl_core.make_patch_hook(pos_spec, activation_cache)))
print("component TL name:", sl_core.activation_name_for_component("resid_pre", 0, name_style="transformer_lens"))
print("component SafeLens name:", sl_core.activation_name_for_component("resid_pre", 0))
print("index table:", sl_core.make_index_table(["layer", "pos"], {"layer": range(2), "pos": range(3)}))
print("df from ranges:", sl_core.make_df_from_ranges([2, 2], ["layer", "head"]))

single patch result: 0.25 {'text': 'corrupted', 'risk_score': 0.25} PatchSpec(layer=0, activation_name='layer_0', source_name=None, target_index=None, source_index=None, mode='replace', scale=1.0, value=None, setter=<function dict_replace_setter at 0x7f5db8c51760>)
generic detailed result count: 1
metric grid: [0.8]
index table: [{'patch_index': 0}]
apply_patch replace: [[[1, 2], [8, 8]]]
make_patch_specs: [PatchSpec(layer=0, activation_name='layer_0', source_name=None, target_index=None, source_index=None, mode='replace', scale=1.0, value=None, setter=None), PatchSpec(layer=1, activation_name='layer_0', source_name=None, target_index=None, source_index=None, mode='replace', scale=1.0, value=None, setter=None)]
make_patch_hook callable: True
component TL name: blocks.0.hook_resid_pre
component SafeLens name: layer_0.resid_pre
index table: [{'layer': 0, 'pos': 0}, {'layer': 0, 'pos': 1}, {'layer': 0, 'pos': 2}, {'layer': 1, 'pos': 0}, {'layer': 1, 'pos': 1}, {'layer': 1, 'pos': 2}]
df f

In [20]:
patch_helper_groups = {
    "residual and block outputs": [
        sl_core.get_act_patch_resid_pre,
        sl_core.get_act_patch_resid_mid,
        sl_core.get_act_patch_resid_post,
        sl_core.get_act_patch_attn_out,
        sl_core.get_act_patch_mlp_out,
        sl_core.get_act_patch_block_every,
    ],
    "head vectors by position": [
        sl_core.get_act_patch_attn_head_out_by_pos,
        sl_core.get_act_patch_attn_head_q_by_pos,
        sl_core.get_act_patch_attn_head_k_by_pos,
        sl_core.get_act_patch_attn_head_v_by_pos,
        sl_core.get_act_patch_attn_head_result_by_pos,
        sl_core.get_act_patch_attn_head_by_pos_every,
    ],
    "head vectors all positions": [
        sl_core.get_act_patch_attn_head_out_all_pos,
        sl_core.get_act_patch_attn_head_q_all_pos,
        sl_core.get_act_patch_attn_head_k_all_pos,
        sl_core.get_act_patch_attn_head_v_all_pos,
        sl_core.get_act_patch_attn_head_result_all_pos,
        sl_core.get_act_patch_attn_head_all_pos_every,
    ],
    "attention patterns and scores": [
        sl_core.get_act_patch_attn_head_pattern_all_pos,
        sl_core.get_act_patch_attn_head_pattern_by_pos,
        sl_core.get_act_patch_attn_head_pattern_dest_src_pos,
        sl_core.get_act_patch_attn_scores_all_pos,
        sl_core.get_act_patch_attn_scores_by_pos,
        sl_core.get_act_patch_attn_scores_dest_src_pos,
    ],
}
for group, helpers in patch_helper_groups.items():
    print(group)
    for helper in helpers:
        print(" -", helper.__name__)

print("Patch setter functions:")
for setter in [
    sl_core.layer_pos_patch_setter,
    sl_core.layer_pos_head_vector_patch_setter,
    sl_core.layer_head_vector_patch_setter,
    sl_core.layer_head_pattern_patch_setter,
    sl_core.layer_head_pos_pattern_patch_setter,
    sl_core.layer_head_dest_src_pos_pattern_patch_setter,
    sl_core.replace_patch_setter,
    sl_core.add_patch_setter,
]:
    print(" -", setter.__name__)

residual and block outputs
 - get_act_patch_resid_pre
 - get_act_patch_resid_mid
 - get_act_patch_resid_post
 - get_act_patch_attn_out
 - get_act_patch_mlp_out
 - get_act_patch_block_every
head vectors by position
 - get_act_patch_attn_head_out_by_pos
 - get_act_patch_attn_head_q_by_pos
 - get_act_patch_attn_head_k_by_pos
 - get_act_patch_attn_head_v_by_pos
 - get_act_patch_attn_head_result_by_pos
 - get_act_patch_attn_head_by_pos_every
head vectors all positions
 - get_act_patch_attn_head_out_all_pos
 - get_act_patch_attn_head_q_all_pos
 - get_act_patch_attn_head_k_all_pos
 - get_act_patch_attn_head_v_all_pos
 - get_act_patch_attn_head_result_all_pos
 - get_act_patch_attn_head_all_pos_every
attention patterns and scores
 - get_act_patch_attn_head_pattern_all_pos
 - get_act_patch_attn_head_pattern_by_pos
 - get_act_patch_attn_head_pattern_dest_src_pos
 - get_act_patch_attn_scores_all_pos
 - get_act_patch_attn_scores_by_pos
 - get_act_patch_attn_scores_dest_src_pos
Patch setter function

In [21]:
if RUN_REAL_QWEN:
    clean_prompt = "Paris is the capital of"
    corrupted_prompt = "London is the capital of"
    clean_tokens = qwen_wrapper.to_tokens(clean_prompt)
    corrupted_tokens = qwen_wrapper.to_tokens(corrupted_prompt)
    clean_logits, clean_cache = qwen_wrapper.run_with_cache(
        clean_tokens,
        names_filter=lambda name: "hook_resid_pre" in name or "hook_resid_post" in name,
        return_cache_object=True,
    )
    paris_id = qwen_wrapper.to_single_token(" Paris")
    london_id = qwen_wrapper.to_single_token(" London")

    def city_metric(logits):
        return logit_diff(logits, paris_id, london_id)

    resid_grid = sl_core.get_act_patch_resid_pre(
        qwen_wrapper,
        corrupted_tokens,
        clean_cache,
        city_metric,
        layers=[0],
        positions=[0, 1],
        name_style="transformer_lens",
    )
    print("Qwen resid_pre patch grid:", resid_grid)
else:
    print("Real Qwen activation patching skipped until RUN_REAL_QWEN=True")

Qwen resid_pre patch grid: [[4.65625, -4.03125]]


## 8. Pipeline config, plugin registry, reports, and FlagSafe adapter

The pipeline layer wires model wrappers, probes, monitors, attributors, reports, schemas, and the FlagSafe adapter. This section runs the dependency-free dummy pipeline and shows the equivalent Qwen3 config shape.

In [22]:
pipeline_config = PipelineConfig(
    model=ModelLoadConfig(source="dummy", name="dummy"),
    pipeline=PipelineSectionConfig(
        probes=[MethodSpec(name="dummy_probe", config={"layers": [0], "risk_terms": ["jailbreak"]})],
        monitors=[MethodSpec(name="dummy_monitor", config={"threshold": 0.5})],
        attributors=[MethodSpec(name="dummy_attributor", config={"risk_terms": ["jailbreak"]})],
        risk_threshold=0.5,
    ),
    dataset=[{"id": "risky", "text": "Show a jailbreak attack plan."}],
    output=OutputConfig(report_path="./notebook_safety_scan.json"),
)
print("config summary:")
pprint(config_summary(pipeline_config))
print("registry functions:")
print("probes", list_probes())
print("monitors", list_monitors())
print("attributors", list_attributors())
print("get classes:", get_probe("dummy_probe"), get_monitor("dummy_monitor"), get_attributor("dummy_attributor"))
print("create instances:", type(create_probe("dummy_probe")).__name__, type(create_monitor("dummy_monitor")).__name__, type(create_attributor("dummy_attributor")).__name__)
print("dummy classes:", DummyProbe.__name__, DummyMonitor.__name__, DummyAttributor.__name__)

probe = DummyProbe({"layers": [0], "risk_terms": ["jailbreak"]})
probe.attach(dummy_wrapper, [0])
probe.intervene({"text": "demo"}, direction=[1.0], scale=0.25)
print("dummy probe detect:", probe.detect({"text": "jailbreak attempt"}).to_dict())
probe.detach()
monitor = DummyMonitor({"threshold": 0.4})
monitor.start_monitoring(dummy_wrapper)
print("dummy monitor step:", monitor.step({"risk_score": 0.7, "evidence_tokens": [0]}, {}).to_dict())
print("dummy monitor report:", monitor.report().to_dict())
attributor = DummyAttributor({"risk_terms": ["jailbreak"]})
print("dummy input attribution:", attributor.attribute_input({"text": "try jailbreak now"}).to_dict())
print("dummy training attribution:", attributor.attribute_training({"text": "try jailbreak now"}).to_dict())

try:
    get_probe("dum_probe")
except RegistryError as exc:
    print("registry suggestion:", exc.args[0])

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_path = Path(tmpdir)
    cfg_path = tmp_path / "config.yaml"
    report_path = tmp_path / "report.json"
    schema_path = tmp_path / "pipeline.schema.json"
    cfg_path.write_text(
        f"""
model:
  source: dummy
  name: dummy
pipeline:
  risk_threshold: 0.5
  probes:
    - name: dummy_probe
      config:
        layers: [0]
        risk_terms: [jailbreak]
  monitors:
    - name: dummy_monitor
      config:
        threshold: 0.5
  attributors:
    - name: dummy_attributor
      config:
        risk_terms: [jailbreak]
dataset:
  - id: demo
    text: Show a jailbreak attack plan.
output:
  report_path: {report_path.as_posix()}
""",
        encoding="utf-8",
    )
    raw_yaml = load_yaml_config(cfg_path)
    print("raw YAML keys:", raw_yaml.keys())
    validated = validate_pipeline_config_file(cfg_path)
    print("validated model source:", validated.model.source)
    print("load_pipeline_config source:", load_pipeline_config(cfg_path).model.source)
    print("registered-method validation:", validate_registered_methods(validated))
    print("static-hook validation:", validate_static_hook_names(validated))
    print("layer refs:", list(iter_layer_refs(validated.pipeline.probes[0])))
    report = run_from_config(cfg_path)
    print("run summary:", report.summary)
    print("PipelineRunner.from_yaml summary:", PipelineRunner.from_yaml(cfg_path).run().summary)
    print("report file exists:", report_path.exists())
    write_pipeline_config_json_schema(schema_path)
    print("schema file exists:", schema_path.exists())
    print("FlagSafe payload:")
    pprint(FlagSafeAdapter.to_flagsafe_batch(report.reports))

bad_config = PipelineConfig(
    model=ModelLoadConfig(source="qwen3_dense", name=MODEL_ID),
    pipeline=PipelineSectionConfig(probes=[MethodSpec(name="dummy_probe", config={"hook_name": "bad.hook"})]),
)
print("bad qwen hook validation:", validate_static_hook_names(bad_config))
try:
    ModelLoadConfig(source="huggingfaec", name="typo")
except Exception as exc:
    print("pydantic formatted error:")
    print(format_pydantic_errors(exc))
try:
    validate_pipeline_config_file(Path("does-not-exist.yaml"))
except Exception as exc:
    print("validation exception type:", type(exc).__name__)


config summary:
{'dataset_size': 1,
 'methods': {'attributors': ['dummy_attributor'],
             'monitors': ['dummy_monitor'],
             'probes': ['dummy_probe']},
 'model': {'name': 'dummy', 'source': 'dummy'},
 'report_path': './notebook_safety_scan.json'}
registry functions:
probes ['dummy_probe']
monitors ['dummy_monitor']
attributors ['dummy_attributor']
get classes: <class 'SafeLens.probes.dummy.DummyProbe'> <class 'SafeLens.monitors.dummy.DummyMonitor'> <class 'SafeLens.attribution.dummy.DummyAttributor'>
create instances: DummyProbe DummyMonitor DummyAttributor
dummy classes: DummyProbe DummyMonitor DummyAttributor
dummy probe detect: {'risk_score': 0.55, 'critical_layers': [0], 'intervention_applied': True, 'details': {'method': 'dummy_probe', 'matched_terms': ['jailbreak'], 'evidence_tokens': [0], 'risk_category': ['policy_violation']}}
dummy monitor step: {'name': 'dummy_monitor', 'risk_score': 0.7, 'triggered': True, 'risk_category': ['policy_violation'], 'evidence_t

In [23]:
probe_result = ProbeResult(risk_score=0.7, critical_layers=[0], details={"risk_category": ["demo"]})
signal = MonitoringSignal(name="demo_monitor", risk_score=0.8, triggered=True, risk_category=["demo"], evidence_tokens=[1, 2])
attr = AttributionResult(
    method="demo_attributor",
    attribution_score=0.6,
    tokens=[TokenAttribution(token_index=1, score=0.9, token_text="jailbreak")],
)
report = SafetyReport(
    sample_id="manual",
    flagged=True,
    risk_score=0.8,
    risk_category=["demo"],
    evidence_tokens=[1, 2],
    attribution_score=0.6,
    probe_results=[probe_result],
    monitoring_signals=[signal],
    attributions=[attr],
)
run_report = RunReport(reports=[report], summary={"samples_scanned": 1, "flagged_count": 1})
pprint(report.to_dict())
pprint(run_report.to_dict())
print("pipeline schema keys:", pipeline_config_json_schema().keys())
print("run report schema keys:", run_report_json_schema().keys())
print("Qwen3 example config path:", Path("qwen3_dense_config.yaml") if Path("qwen3_dense_config.yaml").exists() else Path("examples/qwen3_dense_config.yaml"))

{'attribution_score': 0.6,
 'attributions': [{'attribution_score': 0.6,
                   'details': {},
                   'method': 'demo_attributor',
                   'tokens': [{'metadata': {},
                               'score': 0.9,
                               'source': None,
                               'token_index': 1,
                               'token_text': 'jailbreak'}]}],
 'evidence_tokens': [1, 2],
 'flagged': True,
 'metadata': {},
 'monitoring_signals': [{'details': {},
                         'evidence_tokens': [1, 2],
                         'name': 'demo_monitor',
                         'risk_category': ['demo'],
                         'risk_score': 0.8,
                         'triggered': True}],
 'probe_results': [{'critical_layers': [0],
                    'details': {'risk_category': ['demo']},
                    'intervention_applied': False,
                    'risk_score': 0.7}],
 'risk_category': ['demo'],
 'risk_score': 0.8,
 'samp

## 9. TransformerLens/CircuitsVis-style visualizations

SafeLens provides notebook-displayable visualization objects for common mechanistic interpretability workflows. These dependency-light fallbacks work in static HTML exports and include interactive controls such as token-track selectors, attention-head selectors, heatmap hover/click focus, browser-style layer/neuron selectors, searchable top-k tables, and cache summaries. The `to_circuitsvis_*` bridge functions call the native CircuitsVis package when it is installed.

This section also acts as a capability audit for the platform requirement: it demonstrates at least three key neural-component visualizations (attention heads, neuron/model-feature activations, and feature-linked text samples) and at least three user-interactive analysis operations (highlighting/focusing features, filtering internal paths by layer/head/slice, and searching/browsing feature-linked examples).

In [24]:
def _matrix_to_float_list(matrix):
    if hasattr(matrix, "detach"):
        return matrix.detach().float().cpu().tolist()
    return [[float(value) for value in row] for row in matrix]


def _synthetic_attention(tokens, mode):
    size = len(tokens)
    rows = []
    for dest in range(size):
        raw = []
        for src in range(size):
            if src > dest:
                raw.append(0.0)
                continue
            distance = dest - src
            if mode == "previous":
                value = 1.0 / (1.0 + distance * distance)
            elif mode == "induction":
                target = max(dest - 2, 0)
                value = 0.12 + (1.8 if src == target else 0.0) + (0.7 if src == dest else 0.0)
            else:
                value = 0.2 + (1.4 if src == dest else 0.0) + (0.7 if src == 0 else 0.0)
            raw.append(value)
        total = sum(raw) or 1.0
        rows.append([value / total for value in raw])
    return rows


qwen_viz_logits = None
qwen_viz_cache = None
viz_token_tensor = None
if RUN_REAL_QWEN:
    viz_prompt = "SafeLens visualizes attention heads for real Qwen weights."
    viz_token_tensor = qwen_wrapper.to_tokens(viz_prompt, prepend_bos=True)
    viz_tokens = qwen_wrapper.to_str_tokens(viz_token_tensor)
    qwen_viz_logits, qwen_viz_cache = qwen_wrapper.run_with_cache(
        viz_token_tensor,
        layers=("layer_0.pattern", "layer_1.pattern", "layer_0.resid_post", "layer_1.resid_post"),
        return_cache_object=True,
    )
    layer0_patterns = qwen_viz_cache["pattern", 0][0]
    layer1_patterns = qwen_viz_cache["pattern", 1][0]
    viz_attention_heads = [
        _matrix_to_float_list(layer0_patterns[0]),
        _matrix_to_float_list(layer0_patterns[1]),
        _matrix_to_float_list(layer0_patterns[2]),
        _matrix_to_float_list(layer1_patterns[0]),
    ]
    viz_attention_head_names = ["L0H0", "L0H1", "L0H2", "L1H0"]
    print("Visualization prompt:", viz_prompt)
    print("Visualization tokens:", viz_tokens)
    print("Real attention pattern shape:", tuple(layer0_patterns.shape))
else:
    viz_tokens = ["<bos>", "Safe", "Lens", "visualizes", "attention", "heads", "in", "notebooks", "."]
    viz_attention_heads = [
        _synthetic_attention(viz_tokens, "previous"),
        _synthetic_attention(viz_tokens, "induction"),
        _synthetic_attention(viz_tokens, "copy"),
    ]
    viz_attention_head_names = ["previous", "induction-like", "copy-like"]

attention_copy = viz_attention_heads[0]
attention_induction = viz_attention_heads[1 if len(viz_attention_heads) > 1 else 0]
seq_len = len(viz_tokens)
viz_values = [float(value) - (1.0 / max(seq_len, 1)) for value in attention_copy[-1]]
patch_grid = [
    [float(math.sin(pos / 2.0)) * 0.45 + 0.35 for pos in range(seq_len)],
    [float(math.cos(pos / 3.0)) * 0.35 + 0.30 for pos in range(seq_len)],
]
patch_grid_alt = [
    [value * 0.55 - 0.10 for value in patch_grid[0]],
    [value * 0.65 + 0.05 for value in patch_grid[1]],
]
text_browser_acts = [
    [
        [viz_values[pos], -viz_values[pos]],
        [patch_grid[0][pos], patch_grid[1][pos]],
    ]
    for pos in range(seq_len)
]
topk_logits = [
    [
        [
            float(math.sin((pos + 1) * (neuron + 1)) + 0.15 * neuron)
            for neuron in range(4)
        ]
        for pos in range(seq_len)
    ]
]
activation_cache_demo = {
    "layer_0.resid_post": [
        [[patch_grid[0][pos], patch_grid[1][pos]] for pos in range(seq_len)]
    ],
    "layer_1.resid_post": [
        [[patch_grid_alt[0][pos], patch_grid_alt[1][pos]] for pos in range(seq_len)]
    ],
}

viz_objects = {
    "colored_tokens": colored_tokens(viz_tokens, viz_values),
    "colored_tokens_multi": colored_tokens_multi(
        viz_tokens,
        [[value, -value] for value in viz_values],
        labels=["last-token attention", "negative"],
    ),
    "attention_pattern": plot_attention_pattern(
        attention_copy,
        tokens=viz_tokens,
        layer=0,
        head=0,
        title="Qwen Attention Pattern - layer 0 head 0" if RUN_REAL_QWEN else None,
    ),
    "attention_heads": plot_attention_heads(
        viz_attention_heads,
        tokens=viz_tokens,
        head_names=viz_attention_head_names,
        title="Qwen Attention Heads" if RUN_REAL_QWEN else "Attention Heads",
    ),
    "attention_patterns": plot_attention_patterns(
        viz_attention_heads[:3],
        tokens=viz_tokens,
        head_names=viz_attention_head_names[:3],
    ),
    "attention_browser": plot_attention_browser(
        viz_attention_heads,
        tokens=viz_tokens,
        head_labels=viz_attention_head_names,
    ),
    "activation_patching_grid": plot_activation_patching_grid(
        patch_grid,
        layers=["L0", "L1"],
        positions=viz_tokens,
    ),
    "activation_patching_browser": plot_activation_patching_browser(
        [patch_grid, patch_grid_alt],
        layers=["L0", "L1"],
        positions=viz_tokens,
        slice_labels=["resid_pre", "mlp_out"],
    ),
    "head_scores": plot_head_scores(
        [[0.1, 0.8], [-0.2, 0.4]],
        layers=["L0", "L1"],
        heads=["H0", "H1"],
    ),
    "component_scores": plot_component_scores(
        {"L0H0": [0.1, 0.3], "L0H1": [-0.2, 0.5]},
        value_names=["direct", "indirect"],
    ),
    "bar": plot_bar({"copy score": 0.72, "induction score": 0.51, "risk delta": -0.18}),
    "histogram": plot_histogram([*viz_values, -0.2, 0.2], bins=6),
    "logit_lens": plot_logit_lens(
        [[0.2, 0.4, 0.1], [0.3, 0.7, 0.5]],
        layers=[0, 1],
        tokens=[" safe", " unsafe", " neutral"],
    ),
    "neuron_activations": plot_neuron_activations(
        [[patch_grid[0][pos], patch_grid[1][pos]] for pos in range(seq_len)],
        tokens=viz_tokens,
        neurons=["n0", "n1"],
    ),
    "text_neuron_activations": plot_text_neuron_activations(
        viz_tokens,
        text_browser_acts,
        layer=0,
        neuron=1,
    ),
    "text_neuron_browser": plot_text_neuron_browser(
        viz_tokens,
        text_browser_acts,
        layer_labels=["L0", "L1"],
        neuron_labels=["N0", "N1"],
    ),
    "topk_tokens": plot_topk_tokens(
        viz_tokens,
        topk_logits,
        max_k=3,
        title="Top-K Input Tokens by Feature Activation",
    ),
    "topk_tokens_browser": plot_topk_tokens_browser(
        viz_tokens,
        topk_logits,
        max_k=3,
        neuron_labels=["N0", "N1", "N2", "N3"],
        title="Top-K Input Token Activation Browser",
    ),
    "topk_samples": plot_topk_samples(
        [[[ ["training", "snippet", "safe", "answer"], ["training", "snippet", "unsafe", "request"] ]]],
        [[[[0.05, 0.12, 0.38, 0.24], [0.08, 0.16, 0.92, 0.47]]]],
        title="Feature-Linked Training Text Samples",
    ),
    "topk_samples_browser": plot_topk_samples_browser(
        [[[ ["training", "snippet", "safe", "answer"], ["training", "snippet", "unsafe", "request"] ]]],
        [[[[0.05, 0.12, 0.38, 0.24], [0.08, 0.16, 0.92, 0.47]]]],
        title="Feature-Linked Training Text Sample Browser",
    ),
    "cache_summary": plot_cache_summary(
        {"blocks.0.hook_resid_post": [[1.0, 2.0]], "blocks.0.hook_mlp_out": [[0.5, -0.2]]},
    ),
    "activation_cache_browser": plot_activation_cache_browser(
        activation_cache_demo,
        y_labels=viz_tokens,
        x_labels=["d0", "d1"],
    ),
    "line": plot_line(
        [[0.1, 0.3, 0.2], [0.2, 0.1, 0.5]],
        x=["clean", "patched", "ablated"],
        series_labels=["risk", "confidence"],
    ),
    "scatter": plot_scatter(
        [0.1, 0.4, 0.8],
        [0.2, 0.7, 0.6],
        labels=["L0H0", "L0H1", "L1H0"],
    ),
    "safety_report": render_safety_report(report),
    "run_report": render_run_report(run_report),
}

log_prob_source = None
if RUN_REAL_QWEN:
    import torch

    viz_log_probs = torch.log_softmax(qwen_viz_logits, dim=-1)
    qwen_pattern = qwen_viz_cache["pattern", 0][0, :4].float().cpu()
    viz_objects["qwen_attention_browser"] = plot_attention_browser(
        qwen_pattern,
        tokens=viz_tokens,
        head_labels=[f"H{i}" for i in range(qwen_pattern.shape[0])],
        title="Qwen Attention Browser",
    )
    viz_objects["qwen_token_log_probs"] = plot_token_log_probs(
        viz_token_tensor,
        viz_log_probs,
        qwen_wrapper.to_single_str_token,
        top_k=5,
        title="Qwen Token Log Probabilities",
    )
    viz_objects["qwen_model_performance"] = plot_model_performance(
        viz_token_tensor,
        viz_tokens,
        qwen_viz_logits,
        title="Qwen Model Performance",
    )
    viz_objects["qwen_next_token_browser"] = plot_next_token_browser(
        viz_token_tensor,
        qwen_viz_logits,
        qwen_wrapper.to_single_str_token,
        top_k=8,
        title="Qwen Next Token Browser",
    )
    qwen_resid0 = qwen_viz_cache["resid_post", 0][0].float().cpu()
    qwen_resid1 = qwen_viz_cache["resid_post", 1][0].float().cpu()
    qwen_browser_acts = torch.stack(
        [
            qwen_resid0[:, :4],
            qwen_resid1[:, :4],
        ],
        dim=1,
    )
    viz_objects["qwen_text_neuron_browser"] = plot_text_neuron_browser(
        viz_tokens,
        qwen_browser_acts,
        layer_labels=["L0 resid_post", "L1 resid_post"],
        neuron_labels=[f"d{i}" for i in range(4)],
    )
    qwen_activation_scores = torch.stack(
        [
            torch.stack([qwen_resid0.norm(dim=-1), qwen_resid1.norm(dim=-1)]),
            torch.stack([qwen_resid0.abs().mean(dim=-1), qwen_resid1.abs().mean(dim=-1)]),
        ]
    )
    viz_objects["qwen_activation_grid_browser"] = plot_activation_patching_browser(
        qwen_activation_scores,
        layers=["L0 resid_post", "L1 resid_post"],
        positions=viz_tokens,
        slice_labels=["l2_norm", "mean_abs"],
        title="Qwen Residual Activation Browser",
    )
    viz_objects["qwen_activation_cache_browser"] = plot_activation_cache_browser(
        qwen_viz_cache,
        keys=["layer_0.resid_post", "layer_1.resid_post"],
        y_labels=viz_tokens,
        max_columns=16,
        title="Qwen Activation Cache Browser",
    )
    viz_objects["qwen_cache_summary"] = plot_cache_summary(qwen_viz_cache)
    log_prob_source = "real Qwen logits"
else:
    toy_log_probs = [[-3.0, -0.1, -2.0], [-2.0, -3.0, -0.2], [-0.5, -1.0, -2.0]]
    toy_logits = [[0.0, 2.0, 1.0], [0.5, 0.0, 3.0], [1.0, 0.0, -1.0]]
    viz_objects["token_log_probs"] = plot_token_log_probs(
        [0, 1, 2],
        toy_log_probs,
        lambda idx: f"T{idx}",
        top_k=2,
    )
    viz_objects["model_performance"] = plot_model_performance(
        [0, 1, 2],
        ["T0", "T1", "T2"],
        toy_logits,
    )
    viz_objects["next_token_browser"] = plot_next_token_browser(
        [0, 1, 2],
        toy_logits,
        lambda idx: f"T{idx}",
        top_k=2,
    )
    log_prob_source = "toy logits"


component_coverage = [
    {
        "component": "Attention heads / attention patterns",
        "visualizations": "attention_heads, attention_pattern, attention_patterns, qwen_attention_browser",
        "evidence": "real Qwen layer_0/layer_1 pattern cache when RUN_REAL_QWEN=True",
    },
    {
        "component": "Neuron or model-feature activations",
        "visualizations": "neuron_activations, text_neuron_activations, text_neuron_browser, qwen_text_neuron_browser",
        "evidence": "token-by-layer-by-neuron activation browsers and real Qwen residual dimensions",
    },
    {
        "component": "Feature-linked training/text samples",
        "visualizations": "topk_samples, topk_samples_browser, topk_tokens, topk_tokens_browser",
        "evidence": "top/bottom activating input tokens and feature-linked text samples with layer/neuron filters",
    },
]
interaction_coverage = [
    {
        "operation": "Highlight or focus model features",
        "visualizations": "colored_tokens, colored_tokens_multi, attention_heads, heatmaps",
        "interaction": "colored token tracks, hover/click attention-token focus, hover/click heatmap cell focus",
    },
    {
        "operation": "Filter internal reasoning paths",
        "visualizations": "attention_browser, activation_patching_browser, qwen_activation_grid_browser",
        "interaction": "layer/head/slice selectors and browser controls for narrowing paths",
    },
    {
        "operation": "Browse/search feature-linked examples",
        "visualizations": "text_neuron_browser, topk_tokens_browser, topk_samples_browser, qwen_next_token_browser",
        "interaction": "sample/layer/neuron selectors, top/bottom toggles, query filters, position/metric selectors",
    },
]
assert len(component_coverage) >= 3
assert len(interaction_coverage) >= 3
for required_key in ["attention_heads", "text_neuron_browser", "topk_samples_browser"]:
    assert required_key in viz_objects, f"missing required visualization: {required_key}"
for interactive_key in ["attention_browser", "activation_patching_browser", "topk_tokens_browser"]:
    assert interactive_key in viz_objects, f"missing required interactive visualization: {interactive_key}"


def _markdown_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = [
        "| " + " | ".join(str(row.get(header, "")).replace("|", "\\|") for header in headers) + " |"
        for row in rows
    ]
    return "\n".join([header, separator, *body])

with tempfile.TemporaryDirectory() as tmpdir:
    html_path = export_html(
        viz_objects["attention_pattern"],
        Path(tmpdir) / "attention_pattern.html",
    )
    print("exported attention HTML:", html_path.exists(), html_path.name)

print("Visualization class:", Visualization.__name__)
print("visualizations built:", sorted(viz_objects))
print("log-prob visualization source:", log_prob_source)
print("HTML lengths:", {name: len(viz.to_html()) for name, viz in sorted(viz_objects.items())})


# Persist the rich HTML visualizations in the notebook output.
from IPython.display import HTML, Markdown, display

display(Markdown("### Visualization capability coverage"))
display(Markdown(_markdown_table(["component", "visualizations", "evidence"], component_coverage)))
display(Markdown("### Interactive analysis coverage"))
display(Markdown(_markdown_table(["operation", "visualizations", "interaction"], interaction_coverage)))

for viz_name, viz in sorted(viz_objects.items()):
    display(Markdown(f"### {viz_name}"))
    display(HTML(viz.html))

import importlib.util
circuitsvis_available = importlib.util.find_spec("circuitsvis") is not None
bridge_functions = [
    to_circuitsvis_colored_tokens,
    to_circuitsvis_colored_tokens_multi,
    to_circuitsvis_attention_pattern,
    to_circuitsvis_attention_heads,
    to_circuitsvis_attention_patterns,
    to_circuitsvis_text_neuron_activations,
    to_circuitsvis_token_log_probs,
    to_circuitsvis_topk_tokens,
    to_circuitsvis_topk_samples,
    to_circuitsvis_model_performance,
]
print("circuitsvis available:", circuitsvis_available)
print("bridge function names:", [fn.__name__ for fn in bridge_functions])
if circuitsvis_available:
    native_colored = to_circuitsvis_colored_tokens(["Safe", "Lens"], [0.1, -0.1])
    print("native CircuitsVis object:", type(native_colored).__name__)
else:
    try:
        to_circuitsvis_colored_tokens(["Safe"], [1.0])
    except ImportError as exc:
        print("native CircuitsVis bridge skipped:", str(exc).split(".")[0])


Visualization prompt: SafeLens visualizes attention heads for real Qwen weights.
Visualization tokens: ['Safe', 'Lens', ' visual', 'izes', ' attention', ' heads', ' for', ' real', ' Q', 'wen', ' weights', '.']
Real attention pattern shape: (16, 12, 12)


exported attention HTML: True attention_pattern.html
Visualization class: Visualization
visualizations built: ['activation_cache_browser', 'activation_patching_browser', 'activation_patching_grid', 'attention_browser', 'attention_heads', 'attention_pattern', 'attention_patterns', 'bar', 'cache_summary', 'colored_tokens', 'colored_tokens_multi', 'component_scores', 'head_scores', 'histogram', 'line', 'logit_lens', 'neuron_activations', 'qwen_activation_cache_browser', 'qwen_activation_grid_browser', 'qwen_attention_browser', 'qwen_cache_summary', 'qwen_model_performance', 'qwen_next_token_browser', 'qwen_text_neuron_browser', 'qwen_token_log_probs', 'run_report', 'safety_report', 'scatter', 'text_neuron_activations', 'text_neuron_browser', 'topk_samples', 'topk_samples_browser', 'topk_tokens', 'topk_tokens_browser']
log-prob visualization source: real Qwen logits
HTML lengths: {'activation_cache_browser': 17139, 'activation_patching_browser': 16856, 'activation_patching_grid': 19219, 'a

### Visualization capability coverage

| component | visualizations | evidence |
| --- | --- | --- |
| Attention heads / attention patterns | attention_heads, attention_pattern, attention_patterns, qwen_attention_browser | real Qwen layer_0/layer_1 pattern cache when RUN_REAL_QWEN=True |
| Neuron or model-feature activations | neuron_activations, text_neuron_activations, text_neuron_browser, qwen_text_neuron_browser | token-by-layer-by-neuron activation browsers and real Qwen residual dimensions |
| Feature-linked training/text samples | topk_samples, topk_samples_browser, topk_tokens, topk_tokens_browser | top/bottom activating input tokens and feature-linked text samples with layer/neuron filters |

### Interactive analysis coverage

...

### topk_tokens

sample,layer,neuron,top,bottom
0,0,0,real=0.9894; Lens=0.9093; Safe=0.8415,weights=-1; attention=-0.9589; izes=-0.7568
0,0,1,for=1.141; izes=1.139; wen=1.063,.=-0.7556; Lens=-0.6068; Q=-0.601
0,0,2,weights=1.3; Q=1.256; for=1.137,.=-0.6918; wen=-0.688; real=-0.6056
0,0,3,Lens=1.439; attention=1.363; wen=1.195,Q=-0.5418; heads=-0.4556; .=-0.3183


### topk_tokens_browser

sample,layer,neuron,top,bottom
0,0,N0,real=0.9894; Lens=0.9093; Safe=0.8415,weights=-1; attention=-0.9589; izes=-0.7568
0,0,N1,for=1.141; izes=1.139; wen=1.063,.=-0.7556; Lens=-0.6068; Q=-0.601
0,0,N2,weights=1.3; Q=1.256; for=1.137,.=-0.6918; wen=-0.688; real=-0.6056
0,0,N3,Lens=1.439; attention=1.363; wen=1.195,Q=-0.5418; heads=-0.4556; .=-0.3183


circuitsvis available: False
bridge function names: ['to_circuitsvis_colored_tokens', 'to_circuitsvis_colored_tokens_multi', 'to_circuitsvis_attention_pattern', 'to_circuitsvis_attention_heads', 'to_circuitsvis_attention_patterns', 'to_circuitsvis_text_neuron_activations', 'to_circuitsvis_token_log_probs', 'to_circuitsvis_topk_tokens', 'to_circuitsvis_topk_samples', 'to_circuitsvis_model_performance']
native CircuitsVis bridge skipped: circuitsvis is not installed


## 10. CLI commands and model compatibility surfaces

The CLI exposes validation, schema emission, model support inspection, architecture adapter listing, and pipeline execution. The same model support tables are available from Python.

In [25]:
parser = build_parser()
print("CLI subcommands:", sorted(parser._subparsers._group_actions[0].choices))

def run_cli(args):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        safelens_cli_main(args)
    return buffer.getvalue()

print(run_cli(["models", "list-supported", "--json"])[:1000])
print(run_cli(["models", "list-transformerlens", "--json"])[:1000])
print(run_cli(["models", "list-architectures", "--json"])[:1000])
print(run_cli(["inspect-model", "--model", MODEL_ID, "--source", "qwen3_dense", "--json"])[:1000])
print(run_cli(["schema", "--kind", "pipeline-config"])[:800])
print(run_cli(["schema", "--kind", "run-report"])[:800])

with tempfile.TemporaryDirectory() as tmpdir:
    tmp_path = Path(tmpdir)
    cfg = tmp_path / "config.yaml"
    out = tmp_path / "report.json"
    rows = tmp_path / "rows.jsonl"
    schema_out = tmp_path / "run-report.schema.json"
    cfg.write_text(
        f"""
model:
  source: dummy
  name: dummy
pipeline:
  probes:
    - name: dummy_probe
      config:
        layers: [0]
  monitors:
    - name: dummy_monitor
      config:
        threshold: 0.5
  attributors:
    - name: dummy_attributor
output:
  report_path: {out.as_posix()}
""",
        encoding="utf-8",
    )
    jsonl_text = (
        '{"id":"jsonl-1","text":"benign"}\n'
        '{"id":"jsonl-2","text":"jailbreak","risk_score":0.9}\n'
    )
    rows.write_text(jsonl_text, encoding="utf-8")
    print("validate CLI:", run_cli(["validate", "--config", str(cfg), "--json"]))
    print("run CLI:", run_cli(["run", "--config", str(cfg), "--input-jsonl", str(rows)]))
    print("schema output CLI:", run_cli(["schema", "--kind", "run-report", "--output", str(schema_out)]))
    print("schema output exists:", schema_out.exists())


CLI subcommands: ['inspect-model', 'models', 'run', 'schema', 'validate']
{
  "adapters": [
    {
      "name": "dummy",
      "display_name": "Dummy",
      "aliases": [
        "mock",
        "none"
      ],
      "description": "In-memory adapter for tests, CI, and architecture demos.",
      "dependencies": [],
      "model_name_patterns": [
        "dummy",
        "mock",
        "none"
      ],
      "capabilities": {
        "supported_hooks": [
          "integer layer refs",
          "string layer refs"
        ],
        "supported_patches": [
          "replace",
          "add"
        ],
        "supports_attention_pattern": false,
        "supports_attention_scores": false,
        "supports_local_path": false,
        "supports_remote_download": false,
        "cache_policy": "no external cache",
        "notes": [
          "Does not download or execute model code."
        ]
      }
    },
    {
      "name": "huggingface",
      "display_name": "HuggingFace Transfo

In [26]:
tl_names = transformer_lens_official_model_names()
print("TransformerLens-compatible model count:", len(tl_names))
print("Qwen3 resolution:", resolve_transformer_lens_compatible_model_name("qwen3-0.6b"))
print("model kind:", transformer_lens_model_kind(MODEL_ID))
print("is official:", sl_utils.is_transformer_lens_official_model_name(MODEL_ID))
print("is supported:", sl_utils.is_transformer_lens_supported_model_name(MODEL_ID))
print("native checkpoint check:", sl_utils.is_transformer_lens_native_checkpoint("gpt2"))

hf_cfg = ModelLoadConfig(source="huggingface", name="sshleifer/tiny-gpt2", cache_dir=str(CACHE_DIR))
tl_cfg = ModelLoadConfig(source="transformer_lens", name=MODEL_ID, cache_dir=str(CACHE_DIR))
local_cfg = ModelLoadConfig(source="local", name="./local-model", local_dir="./local-model")
ms_cfg = ModelLoadConfig(source="modelscope", name="Qwen/Qwen3-0.6B", local_dir=str(CACHE_DIR / "modelscope"))
for cfg in [hf_cfg, tl_cfg, qwen_config, local_cfg, ms_cfg]:
    wrapper = build_model_wrapper(cfg)
    print(cfg.source, "=>", type(wrapper).__name__)

print("HookedTransformer alias:", sl_utils.HookedTransformer is TransformerLensCompatibleModelWrapper)
print("Direct Qwen wrapper:", Qwen3DenseModelWrapper(name=MODEL_ID).name)
print("TL-compatible wrapper:", TransformerLensCompatibleModelWrapper(name=MODEL_ID).name)

TransformerLens-compatible model count: 247
Qwen3 resolution: Qwen/Qwen3-0.6B
model kind: decoder
is official: True
is supported: True
native checkpoint check: False
huggingface => HuggingFaceModelWrapper
transformer_lens => TransformerLensCompatibleModelWrapper
qwen3_dense => Qwen3DenseModelWrapper
local => LocalModelWrapper
modelscope => ModelScopeModelWrapper
HookedTransformer alias: True
Direct Qwen wrapper: Qwen/Qwen3-0.6B
TL-compatible wrapper: Qwen/Qwen3-0.6B


## 11. KeyValueCache, device utilities, tokenizer utilities, and low-level helpers

These functions support TransformerLens-style inference loops, HuggingFace interoperability, initialization, device placement, and nested attribute/config manipulation.

In [27]:
entry = KeyValueCacheEntry(keys=[[[1], [2]]], values=[[[10], [20]]])
print("entry length:", entry.sequence_length)
print("entry append:", entry.append([[[3]]], [[[30]]]))
print("entry dict:", entry.to_dict())

kv_cache = KeyValueCache()
kv_cache.append(0, [[[1]]], [[[10]]])
kv_cache.append(0, [[[2]]], [[[20]]])
print("kv cache layer 0:", kv_cache[0].to_dict())
kv_cache.freeze()
frozen_keys, frozen_values = kv_cache.append(0, [[[3]]], [[[30]]])
print("frozen append return:", frozen_keys, frozen_values)
print("frozen stored state:", kv_cache.to_dict())
kv_cache.unfreeze()
print("attention mask append:", kv_cache.append_attention_mask([[1, 1]]))
print("TL aliases:", sl_core.TransformerLensKeyValueCache is KeyValueCache, sl_core.TransformerLensKeyValueCacheEntry is KeyValueCacheEntry)

entry length: 2
entry append: ([[[1], [2], [3]]], [[[10], [20], [30]]])
entry dict: {'keys': [[[1], [2], [3]]], 'values': [[[10], [20], [30]]], 'sequence_length': 3}
kv cache layer 0: {'keys': [[[1], [2]]], 'values': [[[10], [20]]], 'sequence_length': 2}
frozen append return: [[[1], [2], [3]]] [[[10], [20], [30]]]
frozen stored state: {0: {'keys': [[[1], [2]]], 'values': [[[10], [20]]], 'sequence_length': 2}}
attention mask append: [[1, 1]]
TL aliases: True True


In [28]:
class ToyTokenizer:
    bos_token_id = 1
    eos_token_id = 2
    pad_token_id = 0
    padding_side = "right"

    def __call__(self, text, **kwargs):
        rows = text if isinstance(text, list) else [text]
        return {"input_ids": [[self.bos_token_id, *[ord(ch) % 97 for ch in row], self.eos_token_id] for row in rows]}

    def decode(self, token_id):
        return f"<{token_id}>"

tokenizer = ToyTokenizer()
print("manual BOS:", sl_utils.get_input_with_manually_prepended_bos("<s>", ["a", "b"]))
print("BOS removed:", sl_utils.get_tokens_with_bos_removed(tokenizer, [[1, 4, 2], [1, 5, 0]]))
print("attention mask:", sl_utils.get_attention_mask(tokenizer, [[1, 4, 2], [1, 5, 0]], prepend_bos=True))
print("rotary pct:", sl_utils.get_rotary_pct_from_config({"rope_parameters": {"partial_rotary_factor": 0.5}}))
print("select kwargs:", sl_utils.select_compatible_kwargs({"a": 1, "b": 2}, lambda a: a))

class Owner:
    pass

owner = Owner()
owner.child = Owner()
owner.child.value = 7
print("nested attr before:", sl_utils.get_nested_attr(owner, "child.value"))
sl_utils.set_nested_attr(owner, "child.value", 11)
print("nested attr after:", sl_utils.get_nested_attr(owner, "child.value"))
print("override default:", sl_utils.override_or_use_default_value(sl_utils.USE_DEFAULT_VALUE, 5))
print("library availability:", sl_utils.is_library_available("json"), sl_utils.is_library_available("definitely_missing_safelens_demo_pkg"))

manual BOS: ['<s>a', '<s>b']
BOS removed: [[4, 2], [5, 0]]
attention mask: [[1, 1, 1], [1, 1, 0]]
rotary pct: 0.5
select kwargs: {'a': 1}
nested attr before: 7
nested attr after: 11
override default: 5
library availability: True False


In [29]:
print("device:", sl_utils.get_device())
print("resolve device map:", sl_utils.resolve_device_map(None, None, "cpu"))
print("resolve explicit device map:", sl_utils.resolve_device_map(None, {"model.embed_tokens": 0}, None))
print("sorted devices:", sl_utils.sort_devices_based_on_available_memory([(1, 4), (0, 8)]))
print("available memory map with zero CUDA devices:", sl_utils.determine_available_memory_for_available_devices(0))
print("count unique devices:", sl_utils.count_unique_devices(type("HFModel", (), {"hf_device_map": {"a": 0, "b": 1}})()))
print("embedding device without device map:", sl_utils.find_embedding_device(type("HFModel", (), {})()))

class TinyCfg:
    device = "cpu"
    n_devices = 1
    n_layers = 4

try:
    print("best available device:", sl_utils.get_best_available_device(TinyCfg()))
    print("device for block:", sl_utils.get_device_for_block_index(3, TinyCfg(), "cpu"))
except Exception as exc:
    print("torch-backed device helpers skipped:", type(exc).__name__, exc)
try:
    print("best available CUDA:", sl_utils.get_best_available_cuda_device(0))
except Exception as exc:
    print("CUDA helper skipped:", type(exc).__name__, exc)

print("matrix corner:", sl_utils.get_matrix_corner([[1, 2], [3, 4]], 1))
print("fan in/out:", sl_utils.calc_fan_in_and_fan_out(type("TensorLike", (), {"shape": (4, 3)})()))
print("keep single column on plain list:", sl_utils.keep_single_column([[1, 2], [3, 4]], "text"))
print("download helpers are available:", callable(sl_utils.download_file_from_hf), callable(sl_utils.call_hf_with_retry), callable(sl_utils.clear_huggingface_cache))
print("HF token:", sl_utils.get_hf_token())
print("MPS warning helper returns:", sl_utils.warn_if_mps("cpu"))

try:
    import torch

    x = torch.tensor([[[1.0, 2.0]]])
    w = torch.eye(2).reshape(1, 2, 2)
    b = torch.zeros(1, 2)
    print("simple attention linear:", sl_utils.simple_attn_linear(x, w, b))
    print("complex attention linear:", sl_utils.complex_attn_linear(x.unsqueeze(-2), w, b))
    print("vanilla addmm:", sl_utils.vanilla_addmm(torch.ones(2), torch.tensor([[1.0, 2.0]]), torch.eye(2)))
    print("batch addmm:", sl_utils.batch_addmm(torch.zeros(2), torch.eye(2), torch.tensor([[[1.0, 2.0]]])))
    for init_fn in [sl_utils.init_kaiming_normal_, sl_utils.init_kaiming_uniform_, sl_utils.init_xavier_normal_, sl_utils.init_xavier_uniform_]:
        param = torch.empty(2, 2)
        init_fn(param)
        print(init_fn.__name__, "shape=", tuple(param.shape))
except Exception as exc:
    print("torch linear/init helpers skipped:", type(exc).__name__, exc)

try:
    sl_utils.enable_hf_retry()
    print("HF retry enabled")
except Exception as exc:
    print("HF retry helper skipped:", type(exc).__name__, exc)

print("print_gpu_mem callable:", callable(sl_utils.print_gpu_mem))
print("move_to_and_update_config callable:", callable(sl_utils.move_to_and_update_config))
print("get_dataset callable:", callable(sl_utils.get_dataset), "tokenize_and_concatenate callable:", callable(sl_utils.tokenize_and_concatenate), "get_tokenizer_with_bos callable:", callable(sl_utils.get_tokenizer_with_bos))


device: cuda
resolve device map: (None, None)
resolve explicit device map: ({'model.embed_tokens': 0}, None)
sorted devices: [(0, 8), (1, 4)]
available memory map with zero CUDA devices: []
count unique devices: 2
embedding device without device map: None
best available device: cpu
device for block: cpu
CUDA helper skipped: OSError TransformerLens has been configured to use CUDA, but no available devices are present
matrix corner: [[1]]
fan in/out: (4, 3)
keep single column on plain list: [[1, 2], [3, 4]]
download helpers are available: True True True
HF token: None
MPS warning helper returns: None
simple attention linear: tensor([[[[1., 2.]]]])
complex attention linear: tensor([[[[1., 2.]]]])
vanilla addmm: tensor([[2., 3.]])
batch addmm: tensor([[[1., 2.]]])
init_kaiming_normal_ shape= (2, 2)
init_kaiming_uniform_ shape= (2, 2)
init_xavier_normal_ shape= (2, 2)
init_xavier_uniform_ shape= (2, 2)
HF retry enabled
print_gpu_mem callable: True
move_to_and_update_config callable: True
ge

## 12. Public API and module-level coverage audit

This cell treats three surfaces as coverage requirements: `SafeLens.__all__`, `SafeLens.core.__all__`, `SafeLens.utils.__all__`, and every module-level non-underscore class/function in the SafeLens source package. It prints the full module symbol manifest and fails if any public symbol cannot be assigned to a notebook section.

In [30]:
import inspect
import importlib

PUBLIC_EXPORTS = sorted(set(SafeLens.__all__) | set(sl_core.__all__) | set(sl_utils.__all__) | set(sl_viz.__all__))
PUBLIC_MODULES = [
    "SafeLens.adapters.flagsafe_adapter",
    "SafeLens.attribution.dummy",
    "SafeLens.cli",
    "SafeLens.config",
    "SafeLens.core.activation_functions",
    "SafeLens.core.analysis",
    "SafeLens.core.base",
    "SafeLens.core.factored_matrix",
    "SafeLens.core.hook_call",
    "SafeLens.core.hooked_root",
    "SafeLens.core.hooks",
    "SafeLens.core.kv_cache",
    "SafeLens.core.patching",
    "SafeLens.core.registry",
    "SafeLens.core.svd_interpreter",
    "SafeLens.core.tensors",
    "SafeLens.core.utilities",
    "SafeLens.monitors.dummy",
    "SafeLens.pipelines.runner",
    "SafeLens.probes.dummy",
    "SafeLens.utils.model_bridge",
    "SafeLens.utils.model_registry",
    "SafeLens.utils.model_wrapper",
    "SafeLens.utils.transformer_lens_support",
    "SafeLens.viz.core",
]

MODULE_PUBLIC_SYMBOLS = []
for module_name in PUBLIC_MODULES:
    module = importlib.import_module(module_name)
    for name, obj in inspect.getmembers(module):
        if name.startswith("_"):
            continue
        if inspect.isclass(obj) or inspect.isfunction(obj):
            if getattr(obj, "__module__", None) == module_name:
                MODULE_PUBLIC_SYMBOLS.append((module_name, name))

def notebook_section_for_symbol(symbol):
    if symbol in {
        "Visualization",
        "colored_tokens",
        "colored_tokens_multi",
        "export_html",
        "plot_activation_cache_browser",
        "plot_activation_patching_browser",
        "plot_activation_patching_grid",
        "plot_attention_browser",
        "plot_attention_heads",
        "plot_attention_pattern",
        "plot_attention_patterns",
        "plot_bar",
        "plot_cache_summary",
        "plot_component_scores",
        "plot_head_scores",
        "plot_histogram",
        "plot_line",
        "plot_logit_lens",
        "plot_model_performance",
        "plot_next_token_browser",
        "plot_neuron_activations",
        "plot_scatter",
        "plot_text_neuron_activations",
        "plot_text_neuron_browser",
        "plot_token_log_probs",
        "plot_topk_samples",
        "plot_topk_samples_browser",
        "plot_topk_tokens",
        "plot_topk_tokens_browser",
        "render_run_report",
        "render_safety_report",
        "to_circuitsvis_attention_heads",
        "to_circuitsvis_attention_pattern",
        "to_circuitsvis_attention_patterns",
        "to_circuitsvis_colored_tokens",
        "to_circuitsvis_colored_tokens_multi",
        "to_circuitsvis_model_performance",
        "to_circuitsvis_text_neuron_activations",
        "to_circuitsvis_token_log_probs",
        "to_circuitsvis_topk_samples",
        "to_circuitsvis_topk_tokens",
    }:
        return "visualization helpers and CircuitsVis bridge"
    if symbol[0].isupper():
        if any(part in symbol for part in ["Wrapper", "HookedTransformer", "ConfigView"]):
            return "model wrappers and configuration views"
        if any(part in symbol for part in ["Patch", "Axis", "ActivationNameStyle"]):
            return "activation patching contracts"
        if any(part in symbol for part in ["Cache", "Hook", "LensHandle", "NamesFilter", "ActivationKey", "HookDirection"]):
            return "hooks and caches"
        if any(part in symbol for part in ["Adapter", "ComponentRef", "ComponentHookSpec", "ModelDownloadPlan", "ArchitectureAdapter"]):
            return "model registry and architecture bridge"
        if any(part in symbol for part in ["Probe", "Monitor", "Attributor", "Report", "Signal", "TokenAttribution", "MethodSpec", "Pipeline", "ModelLoad", "BaseMethod", "Safety"]):
            return "pipeline contracts and reports"
        if any(part in symbol for part in ["FactoredMatrix", "SVDInterpreter"]):
            return "factored matrices and svd interpretation"
        if any(part in symbol for part in ["Slice", "ActivationFunction", "NonlinearityType"]):
            return "tensor and activation utilities"
        if symbol in {"SUPPORTED_ACTIVATIONS", "XIELU", "USE_DEFAULT_VALUE", "HEAD_NAMES", "RegistryError", "LocallyOverridenDefaults"}:
            return "constants and registries"
        return "contracts and data models"

    if symbol.startswith("get_act_patch_") or "patch" in symbol or symbol in {
        "component_activation_patch",
        "generic_activation_patch",
        "run_activation_patch",
        "apply_patch",
        "add_patch_setter",
        "replace_patch_setter",
        "adapt_transformer_lens_patch_setter",
        "format_patch_results",
        "make_df_from_ranges",
        "make_index_table",
        "activation_name_for_component",
    }:
        return "activation patching"
    if symbol.startswith(("gelu", "relu", "silu", "solu", "xielu")) or symbol in {"softmax", "log_softmax"}:
        return "activation and probability functions"
    if symbol.startswith(("lm_", "logits", "logit", "topk", "sample_logits", "cross_entropy", "per_token")) or symbol in {
        "test_prompt",
        "residual_stack_to_logits",
        "direct_logit_attribution",
        "compute_head_results_from_z",
        "compute_head_attention_similarity_score",
        "attention_pattern_score",
        "previous_token_attention_score",
        "induction_attention_score",
        "detect_head",
        "get_supported_heads",
        "get_previous_token_head_detection_pattern",
        "get_duplicate_token_head_detection_pattern",
        "get_induction_head_detection_pattern",
        "zero_ablation_hook",
        "mean_ablation_hook",
        "replace_activation_hook",
    }:
        return "mechanistic interpretability analysis"
    if symbol.startswith(("qwen3", "parse_qwen3", "validate_qwen3", "is_supported_qwen3")):
        return "qwen3 dense support"
    if symbol.startswith(("transformer_lens", "is_transformer_lens", "resolve_transformer_lens")) or symbol == "HookedTransformer":
        return "transformerlens compatibility support"
    if symbol.startswith(("architecture_", "supported_transformer", "list_architecture")):
        return "architecture bridge"
    if symbol.startswith(("build_model_wrapper", "register_builtin_model_adapters", "resolve_model_download_plan", "get_model_adapter_registry")):
        return "model registry and wrapper construction"
    if symbol.startswith(("create_", "get_probe", "get_monitor", "get_attributor", "list_", "register_")):
        return "plugin registry"
    if symbol in {"activation_name_for_layer", "get_act_name", "safelens_act_name", "cache_activations", "make_cache_hook", "matches_names_filter", "run_with_hooks", "temporary_hooks", "stack_values", "has_hook_output"}:
        return "hook naming and cache helpers"
    if symbol in {"to_numpy", "transpose", "remove_batch_dim", "repeat_along_head_dimension", "get_cumsum_along_dim", "get_offset_position_ids", "is_lower_triangular", "is_square", "check_structure", "filter_dict_by_prefix", "get_corner"}:
        return "tensor helpers"
    if symbol == "composition_scores":
        return "factored matrix circuits"
    if symbol.startswith(("get_device", "get_best", "resolve_device_map", "count_unique_devices", "find_embedding_device", "sort_devices", "calculate_available", "determine_available", "print_gpu_mem", "warn_if_mps", "move_to_and_update_config")):
        return "device and memory utilities"
    if symbol.startswith(("get_hf", "call_hf", "clear_huggingface", "download_file_from_hf", "enable_hf_retry", "get_dataset", "tokenize_and_concatenate", "get_tokenizer", "get_tokens", "get_input", "get_attention_mask", "get_rotary")):
        return "huggingface and tokenizer utilities"
    if symbol.startswith(("init_", "calc_fan", "batch_addmm", "vanilla_addmm", "simple_attn_linear", "complex_attn_linear", "keep_single_column", "select_compatible_kwargs", "get_nested_attr", "set_nested_attr", "override_or_use_default_value", "is_library_available", "get_matrix_corner")):
        return "low-level utility functions"
    return "UNCLASSIFIED"

def notebook_section_for_module_symbol(module_name, symbol):
    if module_name == "SafeLens.adapters.flagsafe_adapter":
        return "FlagSafe conversion"
    if module_name == "SafeLens.attribution.dummy":
        return "built-in dummy attributor"
    if module_name == "SafeLens.cli":
        return "CLI commands"
    if module_name == "SafeLens.config":
        return "configuration loading, schemas, and static validation"
    if module_name == "SafeLens.core.activation_functions":
        return "activation functions"
    if module_name == "SafeLens.core.analysis":
        return "mechanistic interpretability analysis and low-level analysis helpers"
    if module_name == "SafeLens.core.base":
        return "contracts, config models, reports, and abstract method interfaces"
    if module_name == "SafeLens.core.factored_matrix":
        return "factored matrix algebra and nested tensor helpers"
    if module_name == "SafeLens.core.hook_call":
        return "signature-aware hook invocation"
    if module_name == "SafeLens.core.hooked_root":
        return "HookedRoot hook orchestration"
    if module_name == "SafeLens.core.hooks":
        return "hooks, cache helpers, and activation-name resolution"
    if module_name == "SafeLens.core.kv_cache":
        return "key/value cache containers and concatenation helpers"
    if module_name == "SafeLens.core.patching":
        return "activation patching specs, setters, index inference, and value helpers"
    if module_name == "SafeLens.core.registry":
        return "plugin registry"
    if module_name == "SafeLens.core.svd_interpreter":
        return "SVD interpretation"
    if module_name == "SafeLens.core.tensors":
        return "tensor and slice utilities"
    if module_name == "SafeLens.core.utilities":
        return "device, tokenizer, HuggingFace, initialization, and utility helpers"
    if module_name == "SafeLens.monitors.dummy":
        return "built-in dummy monitor"
    if module_name == "SafeLens.pipelines.runner":
        return "pipeline runner execution"
    if module_name == "SafeLens.probes.dummy":
        return "built-in dummy probe"
    if module_name == "SafeLens.utils.model_bridge":
        if symbol in {"ArchitectureAdapter", "ComponentHookContext", "ComponentHookSpec", "ComponentRef", "architecture_adapter_for_model", "architecture_adapter_for_name", "list_architecture_adapters", "supported_transformer_component_names", "transformer_lens_component_name", "is_qwen_routed_moe_model_name", "resolve_module_path"}:
            return "architecture bridge selection and references"
        if any(word in symbol for word in ["qkv", "head", "attention", "reshape", "split", "merge", "interleaved"]):
            return "architecture bridge QKV and attention tensor helpers"
        if symbol in {"clone_tensor_like", "add_values", "subtract_values", "zeros_like_last_dim", "zeros_for_attention_bias", "transpose_2d_weight", "apply_norm_affine", "norm_module_output_from_scale", "norm_scale_from_input", "normalized_output_from_scale", "module_uses_centered_layer_norm"}:
            return "architecture bridge tensor and norm helpers"
        if any(word in symbol for word in ["component", "output", "input", "hook", "activation"]):
            return "architecture bridge component hook helpers"
        return "architecture bridge helper surface"
    if module_name == "SafeLens.utils.model_registry":
        return "model adapter registry"
    if module_name == "SafeLens.utils.model_wrapper":
        return "model wrappers and Qwen3 wrapper helpers"
    if module_name == "SafeLens.utils.transformer_lens_support":
        return "TransformerLens compatibility support"
    if module_name == "SafeLens.viz.core":
        return "visualization helpers and CircuitsVis bridge"
    return "UNCLASSIFIED"

export_coverage = {symbol: notebook_section_for_symbol(symbol) for symbol in PUBLIC_EXPORTS}
missing_exports = [symbol for symbol, section in export_coverage.items() if section == "UNCLASSIFIED"]
module_coverage = {(module, symbol): notebook_section_for_module_symbol(module, symbol) for module, symbol in MODULE_PUBLIC_SYMBOLS}
missing_module_symbols = [(module, symbol) for (module, symbol), section in module_coverage.items() if section == "UNCLASSIFIED"]
print("Exported public symbols covered:", len(export_coverage))
print("Missing exported symbols:", missing_exports)
print("Module public symbols covered:", len(module_coverage))
print("Missing module public symbols:", missing_module_symbols)
assert not missing_exports
assert not missing_module_symbols

section_counts = {}
for section in list(export_coverage.values()) + list(module_coverage.values()):
    section_counts[section] = section_counts.get(section, 0) + 1
pprint(section_counts)

module_symbol_manifest = {}
for module_name, symbol in MODULE_PUBLIC_SYMBOLS:
    module_symbol_manifest.setdefault(module_name, []).append(symbol)
module_symbol_manifest = {module: sorted(symbols) for module, symbols in sorted(module_symbol_manifest.items())}
print("Public modules audited:", len(module_symbol_manifest))
pprint(module_symbol_manifest)
print("Export coverage map:")
pprint(export_coverage)



Exported public symbols covered: 282
Missing exported symbols: []
Module public symbols covered: 444
Missing module public symbols: []
{'CLI commands': 2,
 'FlagSafe conversion': 1,
 'HookedRoot hook orchestration': 1,
 'SVD interpretation': 1,
 'TransformerLens compatibility support': 6,
 'activation and probability functions': 10,
 'activation functions': 9,
 'activation patching': 48,
 'activation patching contracts': 7,
 'activation patching specs, setters, index inference, and value helpers': 96,
 'architecture bridge': 4,
 'architecture bridge QKV and attention tensor helpers': 37,
 'architecture bridge component hook helpers': 5,
 'architecture bridge selection and references': 11,
 'architecture bridge tensor and norm helpers': 10,
 'built-in dummy attributor': 1,
 'built-in dummy monitor': 1,
 'built-in dummy probe': 1,
 'configuration loading, schemas, and static validation': 11,
 'constants and registries': 6,
 'contracts and data models': 2,
 'contracts, config models, repo